## End-to-end agent harness framework

### First phase: minimum component

In [1]:
# initialization
import json
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Markdown
import os
from pprint import pprint

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")
ROOT = Path.cwd()
MODEL = "deepseek-v4-flash"

#### message container

In [2]:
messages = [
    {
        "role":"system",
        "content":f"""
        You are a minimal coding agent running inside Jupyter Notebook.

        Workspace root:
        {ROOT}

        You can use tools when needed.
        Be concise.
        """.strip()
    }
]

In [3]:
pprint(messages)

[{'content': 'You are a minimal coding agent running inside Jupyter Notebook.\n'
             '\n'
             '        Workspace root:\n'
             '        D:\\workflow\\agent_harness\n'
             '\n'
             '        You can use tools when needed.\n'
             '        Be concise.',
  'role': 'system'}]


#### Define one tool

In [404]:
# a tool listing the directory
def list_files(path="."):
    p = (ROOT/path).resolve()

    if not p.is_relative_to(ROOT):
        return "ERROR: path escape workspace"

    if not p.exists():
        return f"ERROR: path does not exist:{path}"
    if not p.is_dir():
        return f"ERROR: not directory: {path}"
    items = []
    for child in sorted(p.iterdir()):
        suffix = "/" if child.is_dir() else ""
        items.append(child.name+suffix)
    return "\n".join(items)

# schema showing to the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory inside the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Directory path relative to workspace root.",
                        "default": "."
                    }
                },
                "required": [],
                "additionalProperties": False,
            },
        },
    }
]

#tool register
tool_impls = {
    "list_files": list_files
}                  

#### Call model

#### Execution the tool use

In [405]:
def execute_tool_call(tool_call):
   name = tool_call["function"]["name"]
   raw_args = tool_call["function"].get("arguments") or "{}"

   try:
       args = json.loads(raw_args)
   except Exception as e:
       return f"ERROR: failed to parse tool arguments: {e}"

   fn = tool_impls.get(name)
   if fn is None:
       return f"ERROR: unknown tool: {name}"

   try:
       return fn(**args)
   except Exception as e:
       return f"ERROR: {type(e).__name__}: {e}"

### Agent Loop

In [406]:
def ask(prompt, max_turns=50):
   messages.append({
       "role": "user",
       "content": prompt,
   })

   for turn in range(max_turns):
       assistant_msg = call_model()
       messages.append(assistant_msg)

       content = assistant_msg.get("content")
       if content:
           display(Markdown(content))

       tool_calls = assistant_msg.get("tool_calls") or []

       if not tool_calls:
           return assistant_msg

       for tool_call in tool_calls:
           name = tool_call["function"]["name"]

           display(Markdown(f"**Tool call:** `{name}`"))

           result = execute_tool_call(tool_call)

           display(Markdown(f"```text\n{result}\n```"))

           messages.append({
               "role": "tool",
               "tool_call_id": tool_call["id"],
               "content": result,
           })

   raise RuntimeError("Agent exceeded max_turns")

### End-to-end test

In [407]:
# ask("请查看当前目录，并告诉我这里大概有什么。")

## Add read_file

### Add path resolver

In [408]:
#add a path resolve avoid tools work outside of the workspace
def resolve_workspace_path(path="."):
   raw = str(path).removeprefix("@")
   p = Path(raw)

   if not p.is_absolute():
       p = ROOT / p

   p = p.resolve()
   root = ROOT.resolve()

   try:
       p.relative_to(root)
   except ValueError:
       raise ValueError(f"path escapes workspace: {path}")

   return p

### add path_resolve in list_files tool

In [409]:
def list_files(path="."):
   p = resolve_workspace_path(path)

   if not p.exists():
       return f"ERROR: path does not exist: {path}"

   if not p.is_dir():
       return f"ERROR: not a directory: {path}"

   items = []
   for child in sorted(p.iterdir()):
       suffix = "/" if child.is_dir() else ""
       items.append(child.name + suffix)

   return "\n".join(items)
#tool re-register
tool_impls["list_files"] = list_files

### read_file

In [410]:
def read_file(path, offset=1, limit=200):
   p = resolve_workspace_path(path)

   if not p.exists():
       return f"ERROR: file does not exist: {path}"

   if not p.is_file():
       return f"ERROR: not a file: {path}"

   text = p.read_text(encoding="utf-8", errors="replace")
   lines = text.splitlines()

   start = max(0, int(offset) - 1)
   end = start + int(limit)

   selected = lines[start:end]

   if not selected:
       return ""

   return "\n".join(
       f"{line_no}: {line}"
       for line_no, line in enumerate(selected, start + 1)
   )

In [411]:
read_file("README.txt", limit=20)

'1: 本项目学习Pi agent的harness框架，从简单到复杂，构建一个类似的python版本，通过实践来学习agent harness'

### add read_file schema

In [412]:
tools.append({
   "type": "function",
   "function": {
       "name": "read_file",
       "description": "Read a text file inside the workspace. Returns numbered lines.",
       "parameters": {
           "type": "object",
           "properties": {
               "path": {
                   "type": "string",
                   "description": "File path relative to workspace root."
               },
               "offset": {
                   "type": "integer",
                   "description": "1-based line number to start reading from.",
                   "default": 1
               },
               "limit": {
                   "type": "integer",
                   "description": "Maximum number of lines to read.",
                   "default": 200
               }
           },
           "required": ["path"],
           "additionalProperties": False,
       },
   },
})

### Register real_file tool

In [413]:
tool_impls["read_file"] = read_file

In [414]:
tool_impls

{'list_files': <function __main__.list_files(path='.')>,
 'read_file': <function __main__.read_file(path, offset=1, limit=200)>}

### End-to-end test

In [415]:
# ask("请查看当前目录，选择一个看起来重要的文本文件，读取它，并总结它的内容。")

## Add write_file tool

### add confirm_tool_call

In [416]:
def confirm_tool_call(name, args):
   print(f"\nTool `{name}` wants to run with args:")
   print(json.dumps(args, ensure_ascii=False, indent=2))

   answer = input("Allow? [y/N] ").strip().lower()
   return answer in ("y", "yes")

### which tool need to confirm

In [417]:
TOOLS_REQUIRE_CONFIRM = {
   "write_file",
}

### revice execute_tool_call

In [418]:
def execute_tool_call(tool_call):
   name = tool_call["function"]["name"]
   raw_args = tool_call["function"].get("arguments") or "{}"

   try:
       args = json.loads(raw_args)
   except Exception as e:
       return f"ERROR: failed to parse tool arguments: {e}"

   fn = tool_impls.get(name)
   if fn is None:
       return f"ERROR: unknown tool: {name}"

   if name in TOOLS_REQUIRE_CONFIRM:
       allowed = confirm_tool_call(name, args)
       if not allowed:
           return "BLOCKED: user rejected tool call"

   try:
       return fn(**args)
   except Exception as e:
       return f"ERROR:{type(e).__name__}:{e}"

### write_file

In [419]:
def write_file(path, content):
   p = resolve_workspace_path(path)

   # 防止把目录当文件写
   if p.exists() and p.is_dir():
       return f"ERROR: path is a directory: {path}"

   p.parent.mkdir(parents=True, exist_ok=True)
   p.write_text(content, encoding="utf-8")

   return f"Wrote file: {path} ({len(content)} chars)"

In [420]:
write_file("tmp-mini-test.txt", "hello")

'Wrote file: tmp-mini-pi-test.txt (5 chars)'

In [421]:
read_file("tmp-mini-test.txt")

'1: hello'

In [422]:
Path("tmp-mini-test.txt").unlink()

### Register the tool

In [423]:
tool_impls["write_file"] = write_file

### add write_file schema

In [424]:
tools.append({
   "type": "function",
   "function": {
       "name": "write_file",
       "description": "Create or overwrite a text file inside the workspace.",
       "parameters": {
           "type": "object",
           "properties": {
               "path": {
                   "type": "string",
                   "description": "File path relative to workspace root."
               },
               "content": {
                   "type": "string",
                   "description": "Full file content to write."
               }
           },
           "required": ["path", "content"],
           "additionalProperties": False,
       },
   },
})

### update system prompt

In [425]:
messages[0]["content"] = f"""
You are a minimal coding agent running inside Jupyter Notebook.

Workspace root:
{ROOT}

Available tools:
- list_files: list files in a directory.
- read_file: read a text file.
- write_file: create or overwrite a text file.

Rules:
- Use list_files/read_file before modifying existing files.
- Use write_file only when creating a new file or replacing a full file.
- Be concise.
""".strip()

### End-to-end test

In [4]:
# ask("xxx")

## Add edit_file

In [427]:
import difflib

### add simple truncate

In [428]:
def truncate_text(text,max_chars = 12000):
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n\n[truncated: {len(text)} chars total]"

### edit_file

In [429]:
def edit_file(path,edits):
    p = resolve_workspace_path(path)

    if not p.exists():
        return f"ERROR: file does not exist: {path}"
    if not p.is_file():
        return f"ERROR: not a file: {path}"

    old = p.read_text(encoding="utf-8", errors = "replace")
    new = old

    for i, edit in enumerate(edits, start=1):
        old_text = edit["oldText"]
        new_text = edit["newText"]

        if old_text == "":
            return f"ERROR: edit #{i} oldText is empty"

        count = new.count(old_text)

        if count == 0:
            return f"ERROR: edit #{i} oldText did not match"
        if count > 1:
            return f"ERROR: eidt #{i} oldText matched {count} times; include more surrounding context"
        new = new.replace(old_text, new_text, 1)

    if new == old:
        return "No changes"
    p.write_text(new, encoding="utf-8")

    diff = "".join(
    difflib.unified_diff(
    old.splitlines(keepends=True),
    new.splitlines(keepends=True),
    fromfile=f"{path} before",
    tofile=f"{path} after",
    )
    )
    return truncate_text(f"Edited file: {path}\n\n{diff}")
    

### Register the tool and need confirmer

In [430]:
tool_impls["edit_file"] = edit_file
TOOLS_REQUIRE_CONFIRM.add("edit_file")

In [431]:
tool_impls

{'list_files': <function __main__.list_files(path='.')>,
 'read_file': <function __main__.read_file(path, offset=1, limit=200)>,
 'write_file': <function __main__.write_file(path, content)>,
 'edit_file': <function __main__.edit_file(path, edits)>}

In [432]:
TOOLS_REQUIRE_CONFIRM

{'edit_file', 'write_file'}

### add edit_file schema

In [433]:
tools.append({
   "type": "function",
   "function": {
       "name": "edit_file",
       "description": (
           "Edit a text file using exact text replacement. "
           "Use this after read_file. Each oldText must match exactly once."
       ),
       "parameters": {
           "type": "object",
           "properties": {
               "path": {
                   "type": "string",
                   "description": "File path relative to workspace root."
               },
               "edits": {
                   "type": "array",
                   "description": "One or more exact text replacements.",
                   "items": {
                       "type": "object",
                       "properties": {
                           "oldText": {
                               "type": "string",
                               "description": "Exact text currently in the file."
                           },
                           "newText": {
                               "type": "string",
                               "description": "Replacement text."
                           }
                       },
                       "required": ["oldText", "newText"],
                       "additionalProperties": False,
                   },
                   "minItems": 1,
               }
           },
           "required": ["path", "edits"],
           "additionalProperties": False,
       },
   },
})

### update system prompt

In [434]:
messages[0]["content"] = f"""
You are a minimal coding agent running inside Jupyter Notebook.

Workspace root:
{ROOT}

Available tools:
- list_files: list files in a directory.
- read_file: read a text file.
- write_file: create or overwrite a text file.
- edit_file: edit a file by exact text replacement.

Rules:
- Use list_files/read_file before modifying existing files.
- Use edit_file for targeted modifications.
- oldText in edit_file must match exactly once.
- If oldText does not match or matches multiple times, read more context and retry.
- Use write_file only when creating a new file or replacing a full file.
- Be concise.
""".strip()

### test

In [435]:
write_file("notes/edit-demo.md", """# Mini Pi

这是一个简单测试。
""")

'Wrote file: notes/edit-demo.md (21 chars)'

In [436]:
read_file("notes/edit-demo.md")

'1: # Mini Pi\n2: \n3: 这是一个简单测试。'

In [437]:
edit_file(
   "notes/edit-demo.md",
   [
       {
           "oldText": "这是一个简单测试。",
           "newText": "这是一个最小端到端测试。"
       }
   ]
)

'Edited file: notes/edit-demo.md\n\n--- notes/edit-demo.md before\n+++ notes/edit-demo.md after\n@@ -1,3 +1,3 @@\n # Mini Pi\n \n-这是一个简单测试。\n+这是一个最小端到端测试。\n'

In [438]:
read_file("notes/edit-demo.md")

'1: # Mini Pi\n2: \n3: 这是一个最小端到端测试。'

### End-to-end test

In [439]:
# ask("请读取 notes/edit-demo.md，把标题 Mini Pi 改成 Mini Pi Agent Harness。")

## Add bash tool

### add to confirm

In [440]:
TOOLS_REQUIRE_CONFIRM.add("bash")

### bash tool

In [441]:
import subprocess
def bash(command, timeout=30):
   timeout = int(timeout)
   timeout = min(timeout, 120)  # 最长 120 秒，避免 notebook 卡死

   dangerous_patterns = [
       "rm -rf /",
       "rm -rf ~",
       "mkfs",
       "dd if=",
       ":(){",
       "sudo ",
       "chmod -R 777",
   ]

   for pattern in dangerous_patterns:
       if pattern in command:
           return f"ERROR: blocked dangerous command: {pattern}"

   try:
       result = subprocess.run(
           command,
           shell=True,
           cwd=ROOT,
           text=True,
           capture_output=True,
           timeout=timeout,
       )
       output = f"""$ {command}
       exit_code: {result.returncode}

       STDOUT:
       {result.stdout}

       STDERR:
       {result.stderr}
       """
       return truncate_text(output, max_chars = 12000)
   except subprocess.TimeoutExpired as e:
       stdout = e.stdout or ""
       stderr = e.stderr or ""
       output = f"""$ {command}
       ERROR: command timed out after {timeout} seconds
       PARTIAL STDOUT:
       {stdout}

       PARTIAL STDERR:
       {stderr}
       """
       return truncate_text(output,max_chars = 12000)
   except Exception as e:
       return f"ERROR: {type(e).__name__}: {e}"


### register the tool

In [442]:
tool_impls["bash"] = bash

In [443]:
tool_impls

{'list_files': <function __main__.list_files(path='.')>,
 'read_file': <function __main__.read_file(path, offset=1, limit=200)>,
 'write_file': <function __main__.write_file(path, content)>,
 'edit_file': <function __main__.edit_file(path, edits)>,
 'bash': <function __main__.bash(command, timeout=30)>}

### bash schema

In [444]:
tools.append({
   "type": "function",
   "function": {
       "name": "bash",
       "description": (
           "Run a shell command in the workspace. "
           "Use for inspection, tests, grep, find, git status, python scripts, etc."
       ),
       "parameters": {
           "type": "object",
           "properties": {
               "command": {
                   "type": "string",
                   "description": "Shell command to run."
               },
               "timeout": {
                   "type": "integer",
                   "description": "Timeout in seconds.",
                   "default": 30
               }
           },
           "required": ["command"],
           "additionalProperties": False,
       },
   },
})

### update system prompt

In [445]:
messages[0]["content"] = f"""
You are a minimal coding agent running inside Jupyter Notebook.

Workspace root:
{ROOT}

Available tools:
- list_files: list files in a directory.
- read_file: read a text file.
- write_file: create or overwrite a text file.
- edit_file: edit a file by exact text replacement.
- bash: run shell commands in the workspace.

Rules:
- Use list_files/read_file before modifying existing files.
- Use edit_file for targeted modifications.
- oldText in edit_file must match exactly once.
- Use write_file only when creating a new file or replacing a full file.
- Use bash for inspection, tests, search, and simple scripts.
- Avoid destructive commands.
- Be concise.
""".strip()

### test

In [446]:
bash("dir", timeout = 10)

'$ dir\n       exit_code: 0\n\n       STDOUT:\n        驱动器 D 中的卷是 Data\n 卷的序列号是 D6C5-93B5\n\n D:\\workflow\\agent_harness 的目录\n\n2026/07/30  14:31    <DIR>          .\n2026/07/05  15:25    <DIR>          ..\n2026/07/16  13:42               139 .gitignore\n2026/07/08  15:53    <DIR>          .ipynb_checkpoints\n2026/07/26  16:13    <DIR>          .mini-pi\n2026/07/08  15:59    <DIR>          .nb-agent\n2026/07/05  18:07           167,770 harness_demo.ipynb\n2026/07/05  17:09    <DIR>          harness_lab\n2026/07/30  14:31           407,261 mini_harness.ipynb\n2026/07/22  16:35    <DIR>          notes\n2026/07/11  23:57               133 README.txt\n               4 个文件        575,303 字节\n               7 个目录 396,548,866,048 可用字节\n\n\n       STDERR:\n       \n       '

### End-to-end test

In [447]:
messages = messages[:1]

In [448]:
# ask("分析mini_harness.ipynb，在开头给出系统架构描述")

In [449]:
len(messages)

1

# build session

### build session directory

In [450]:
import time
import uuid
import json
from pathlib import Path

SESSION_DIR = ROOT / ".mini-pi" / "sessions"
SESSION_DIR.mkdir(parents=True, exist_ok=True)

SESSION_FILE = None
SESSION_ID = None
SESSION_LEAF_ID = None

### JSONL read and write tool

In [451]:
def read_jsonl(path):
   path = Path(path)
   if not path.exists():
       return []

   entries = []
   for line in path.read_text(encoding="utf-8").splitlines():
       if line.strip():
           entries.append(json.loads(line))
   return entries


def write_jsonl_line(path, obj):
   with Path(path).open("a", encoding="utf-8") as f:
       f.write(json.dumps(obj, ensure_ascii=False) + "\n")

### Current system message 

In [452]:
def current_system_message():
   if messages and messages[0].get("role") == "system":
       return dict(messages[0])

   return {
       "role": "system",
       "content": "You are a minimal coding agent running inside Jupyter Notebook."
   }

### new session

In [453]:
def new_session():
   global SESSION_FILE, SESSION_ID, SESSION_LEAF_ID, messages

   SESSION_ID = uuid.uuid4().hex
   SESSION_LEAF_ID = None

   filename = time.strftime("%Y%m%d-%H%M%S") + f"-{SESSION_ID[:8]}.jsonl"
   SESSION_FILE = SESSION_DIR / filename

   system = current_system_message()
   messages = [system]

   header = {
       "type": "session",
       "id": SESSION_ID,
       "timestamp": time.time(),
       "cwd": str(ROOT),
   }

   write_jsonl_line(SESSION_FILE, header)

   print(f"New session: {SESSION_FILE}")
   return SESSION_FILE

In [454]:
new_session()

New session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-90fe342b.jsonl


WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-90fe342b.jsonl')

In [455]:
read_file('D:/workflow/agent_harness/.mini-pi/sessions/20260712-232412-054645d4.jsonl')

'1: {"type": "session", "id": "054645d4a601457e9c4577ff006fc9d6", "timestamp": 1783869852.5980003, "cwd": "D:\\\\workflow\\\\agent_harness"}'

### append session entry

In [456]:
def append_session_entry(entry):
    global SESSION_LEAF_ID
    record = {
        "id": uuid.uuid4().hex[:8],
        "parentId": SESSION_LEAF_ID,
        "timestamp": time.time(),
        **entry,
    }
    SESSION_LEAF_ID = record["id"]
    write_jsonl_line(SESSION_FILE, record)
    return record

def save_message(msg):
    append_session_entry({
        "type": "message",
        "message": msg,
    })
    

### replace ask with the session version 

In [457]:
def ask(prompt, max_turns = 50):
    if SESSION_FILE is None:
        raise RuntimeError("No active session. Run new_session() or resume_session(path) first.")
    user_msg = {
    "role":"user",
    "content":prompt,
    }
    messages.append(user_msg)
    save_message(user_msg)
    for turn in range(max_turns):
        assistant_msg = call_model()
        messages.append(assistant_msg)
        save_message(assistant_msg)
        content = assistant_msg.get("content")
        if content:
            display(Markdown(content))
        tool_calls = assistant_msg.get("tool_calls") or []

        if not tool_calls:
            return assistant_msg
        for tool_call in tool_calls:
            name = tool_call["function"]["name"]
            display(Markdown(f"**Tool call:** `{name}`"))

            result = execute_tool_call(tool_call)

            display(Markdown(f"```text\n{result[:3000]}\n```"))

            tool_msg = {
                "role":"tool",
                "tool_call_id":tool_call["id"],
                "content":result,
            }
            messages.append(tool_msg)
            save_message(tool_msg)

    raise RuntimeError("Agent exceeded max_turns")
        

            

### test

In [458]:
new_session()
ask("请查看当前目录并简单总结")

New session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


**Tool call:** `list_files`

```text
.git/
.gitignore
.ipynb_checkpoints/
.mini-pi/
.nb-agent/
harness_demo.ipynb
harness_lab/
mini_harness.ipynb
notes/
README.txt
```

**Tool call:** `list_files`

```text
benchmark_bigcodebench_sample.json
bigcodebench_hard_candidate_pool.json
bigcodebench_hard_valid_pool.json
canonical_validation_results.csv
runs/
```

...

```text
edit-demo.md
hello-mini-pi.md
permission-test.md
双皮奶做法.md
```

**Tool call:** `read_file`

```text
1: 本项目学习Pi agent的harness框架，从简单到复杂，构建一个类似的python版本，通过实践来学习agent harness
```

当前目录的工作区概要如下：

### 根目录 (`agent_harness`)

| 项目 | 说明 |
|------|------|
| `README.txt` | 项目描述：学习 Pi agent 的 harness 框架，从简单到复杂构建 Python 版本 |
| `harness_demo.ipynb` | 主演示 Notebook |
| `mini_harness.ipynb` | 迷你版 harness Notebook |
| `harness_lab/` | 实验目录，含 BigCodeBench 基准评测数据（JSON 配置 + CSV 结果）和 `runs/` 子目录 |
| `notes/` | 笔记目录（含编辑、mini-pi、权限测试笔记，以及一篇**双皮奶做法** 🍮） |
| `.git/`, `.gitignore` | Git 版本控制 |
| `.ipynb_checkpoints/` | Jupyter 自动检查点 |
| `.mini-pi/`, `.nb-agent/` | 隐藏配置/状态目录 |

**一句话总结**：这是一个通过从零实现 agent harness 来学习 Pi agent 框架的 Python/Jupyter 项目，附带 BigCodeBench 基准测试的实验数据和笔记文档。

{'content': '当前目录的工作区概要如下：\n\n### 根目录 (`agent_harness`)\n\n| 项目 | 说明 |\n|------|------|\n| `README.txt` | 项目描述：学习 Pi agent 的 harness 框架，从简单到复杂构建 Python 版本 |\n| `harness_demo.ipynb` | 主演示 Notebook |\n| `mini_harness.ipynb` | 迷你版 harness Notebook |\n| `harness_lab/` | 实验目录，含 BigCodeBench 基准评测数据（JSON 配置 + CSV 结果）和 `runs/` 子目录 |\n| `notes/` | 笔记目录（含编辑、mini-pi、权限测试笔记，以及一篇**双皮奶做法** 🍮） |\n| `.git/`, `.gitignore` | Git 版本控制 |\n| `.ipynb_checkpoints/` | Jupyter 自动检查点 |\n| `.mini-pi/`, `.nb-agent/` | 隐藏配置/状态目录 |\n\n**一句话总结**：这是一个通过从零实现 agent harness 来学习 Pi agent 框架的 Python/Jupyter 项目，附带 BigCodeBench 基准测试的实验数据和笔记文档。',
 'role': 'assistant',
 'reasoning_content': 'Now I have a good picture of the workspace. Let me summarize.'}

In [459]:
SESSION_FILE

WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-b5733715.jsonl')

In [460]:
read_jsonl(SESSION_FILE)

[{'type': 'session',
  'id': 'b573371535a2456b83312d2d1a12f513',
  'timestamp': 1785393080.5215404,
  'cwd': 'D:\\workflow\\agent_harness'},
 {'id': 'ad3890fc',
  'parentId': None,
  'timestamp': 1785393080.5232275,
  'type': 'message',
  'message': {'role': 'user', 'content': '请查看当前目录并简单总结'}},
 {'id': '10eccdee',
  'parentId': 'ad3890fc',
  'timestamp': 1785393081.950766,
  'type': 'message',
  'message': {'content': '',
   'role': 'assistant',
   'tool_calls': [{'id': 'call_00_GVKVybirqVJLuA4qmzoX2327',
     'function': {'arguments': '{"path": "."}', 'name': 'list_files'},
     'type': 'function',
     'index': 0}],
   'reasoning_content': 'The user wants me to look at the current directory and give a summary. Let me start by listing the files.'}},
 {'id': 'a6389c54',
  'parentId': '10eccdee',
  'timestamp': 1785393081.9584973,
  'type': 'message',
  'message': {'role': 'tool',
   'tool_call_id': 'call_00_GVKVybirqVJLuA4qmzoX2327',
   'content': '.git/\n.gitignore\n.ipynb_checkpoints

### list all session

In [461]:
def message_text(msg):
    content = msg.get("content","")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            x.get("text", "")
            for x in content
            if isinstance(x, dict) and x.get("type") == "text"
        )
    return ""

def list_sessions():
    files = sorted(
        SESSION_DIR.glob("*.jsonl"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for i, path in enumerate(files):
        entries = read_jsonl(path)
        msgs = [
        e["message"]
        for e in entries
        if e.get("type") == "message"
        ]
        first_user = next(
        (message_text(m) for m in msgs if m.get("role") == "user"),
        "",
        )
        print(f"[{i}]{path.name}")
        print(f"  messages:{len(msgs)}")
        print(f"  first:{first_user[:80]}")
        

In [462]:
list_sessions()

[0]20260730-143120-b5733715.jsonl
  messages:8
  first:请查看当前目录并简单总结
[1]20260730-143120-90fe342b.jsonl
  messages:0
  first:
[2]20260730-142022-fork-19228dcd.jsonl
  messages:15
  first:请查看当前目录并简单总结
[3]20260730-141904-fork-dfb9b22e.jsonl
  messages:13
  first:请查看当前目录并简单总结
[4]20260716-131755-8345e18d.jsonl
  messages:198
  first:请查看当前目录并简单总结
[5]20260714-112646-7848e15f.jsonl
  messages:14
  first:请查看当前目录并简单总结
[6]20260714-112553-9a6f27e6.jsonl
  messages:3
  first:请查看当前目录并简单总结
[7]20260712-232412-054645d4.jsonl
  messages:0
  first:


### resume session

In [463]:
def resume_session(path):
    global SESSION_FILE, SESSION_ID, SESSION_LEAF_ID, messages

    SESSION_FILE = Path(path)
    entries = read_jsonl(SESSION_FILE)
    if not entries or entries[0].get("type") != "session":
        raise ValueError("Not a valid mini-pi session file")
    SESSION_ID = entries[0]["id"]

    SESSION_LEAF_ID = None
    for e in entries[1:]:
        if "id" in e:
            SESSION_LEAF_ID = e["id"]
    system = current_system_message()
    restored = [system]
    for e in entries[1:]:
        if e.get("type") == "message":
            restored.append(e["message"])
    messages = restored
    print(f"Resumed session: {SESSION_FILE}")
    print(f"Restored messages: {len(messages)}")
    

In [464]:
read_jsonl(SESSION_FILE)

[{'type': 'session',
  'id': 'b573371535a2456b83312d2d1a12f513',
  'timestamp': 1785393080.5215404,
  'cwd': 'D:\\workflow\\agent_harness'},
 {'id': 'ad3890fc',
  'parentId': None,
  'timestamp': 1785393080.5232275,
  'type': 'message',
  'message': {'role': 'user', 'content': '请查看当前目录并简单总结'}},
 {'id': '10eccdee',
  'parentId': 'ad3890fc',
  'timestamp': 1785393081.950766,
  'type': 'message',
  'message': {'content': '',
   'role': 'assistant',
   'tool_calls': [{'id': 'call_00_GVKVybirqVJLuA4qmzoX2327',
     'function': {'arguments': '{"path": "."}', 'name': 'list_files'},
     'type': 'function',
     'index': 0}],
   'reasoning_content': 'The user wants me to look at the current directory and give a summary. Let me start by listing the files.'}},
 {'id': 'a6389c54',
  'parentId': '10eccdee',
  'timestamp': 1785393081.9584973,
  'type': 'message',
  'message': {'role': 'tool',
   'tool_call_id': 'call_00_GVKVybirqVJLuA4qmzoX2327',
   'content': '.git/\n.gitignore\n.ipynb_checkpoints

In [465]:
resume_session(SESSION_FILE)

Resumed session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
Restored messages: 9


In [466]:
read_file(SESSION_FILE)

'1: {"type": "session", "id": "b573371535a2456b83312d2d1a12f513", "timestamp": 1785393080.5215404, "cwd": "D:\\\\workflow\\\\agent_harness"}\n2: {"id": "ad3890fc", "parentId": null, "timestamp": 1785393080.5232275, "type": "message", "message": {"role": "user", "content": "请查看当前目录并简单总结"}}\n3: {"id": "10eccdee", "parentId": "ad3890fc", "timestamp": 1785393081.950766, "type": "message", "message": {"content": "", "role": "assistant", "tool_calls": [{"id": "call_00_GVKVybirqVJLuA4qmzoX2327", "function": {"arguments": "{\\"path\\": \\".\\"}", "name": "list_files"}, "type": "function", "index": 0}], "reasoning_content": "The user wants me to look at the current directory and give a summary. Let me start by listing the files."}}\n4: {"id": "a6389c54", "parentId": "10eccdee", "timestamp": 1785393081.9584973, "type": "message", "message": {"role": "tool", "tool_call_id": "call_00_GVKVybirqVJLuA4qmzoX2327", "content": ".git/\\n.gitignore\\n.ipynb_checkpoints/\\n.mini-pi/\\n.nb-agent/\\nharness_

In [467]:
# ask("基于刚才的上下文，总结你知道的信息")

## minimum command system + auto resume recent session

### obtain session list

In [468]:
def get_session_files():
    return sorted(
        SESSION_DIR.glob("*.jsonl"),
        key=lambda p: p.stat().st_mtime,
        reverse = True,
    )
def latest_session():
    files = get_session_files()
    return files[0] if files else None

### improve list_sessions

In [469]:
def list_sessions():
    files = get_session_files()
    if not files:
        print("No sessions.")
        return []

    for i, path in enumerate(files):
        entries = read_jsonl(path)
        msgs = [
           e["message"]
           for e in entries
           if e.get("type") == "message"
        ]
        
        first_user = next(
           (message_text(m) for m in msgs if m.get("role") == "user"),
           "",
        )
        
        print(f"[{i}] {path.name}")
        print(f"    messages: {len(msgs)}")
        print(f"    first: {first_user[:100]}")

    return files
    

In [470]:
list_sessions()

[0] 20260730-143120-b5733715.jsonl
    messages: 8
    first: 请查看当前目录并简单总结
[1] 20260730-143120-90fe342b.jsonl
    messages: 0
    first: 
[2] 20260730-142022-fork-19228dcd.jsonl
    messages: 15
    first: 请查看当前目录并简单总结
[3] 20260730-141904-fork-dfb9b22e.jsonl
    messages: 13
    first: 请查看当前目录并简单总结
[4] 20260716-131755-8345e18d.jsonl
    messages: 198
    first: 请查看当前目录并简单总结
[5] 20260714-112646-7848e15f.jsonl
    messages: 14
    first: 请查看当前目录并简单总结
[6] 20260714-112553-9a6f27e6.jsonl
    messages: 3
    first: 请查看当前目录并简单总结
[7] 20260712-232412-054645d4.jsonl
    messages: 0
    first: 


[WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-b5733715.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-90fe342b.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-142022-fork-19228dcd.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-141904-fork-dfb9b22e.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260716-131755-8345e18d.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260714-112646-7848e15f.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260714-112553-9a6f27e6.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260712-232412-054645d4.jsonl')]

### auto resume recent session

In [471]:
def auto_resume_or_new():
    latest = latest_session()

    if latest:
        print(f"Auto-resuming latest session.")
        print(latest)
        resume_session(latest)
    else:
        print("No existing session. Creating a new one.")
        new_session()
        

In [472]:
auto_resume_or_new()

Auto-resuming latest session.
D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
Resumed session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
Restored messages: 9


### command processor

In [473]:
def handle_command(text):
    global MODEL

    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""

    if cmd in ("help","?"):
        print("""
       Available commands:
   /help              Show commands
   /new               Start a new session
   /sessions          List sessions
   /resume [index]    Resume session by index or path
   /state             Show current state
   /model [name]      Show or set model
   /tools             Show available tools
   """.strip())
        return
    if cmd == "new":
        return new_session()
    if cmd == "sessions":
        return list_sessions()
    if cmd == "resume":
        files = list_sessions()
        if not files:
            return
        if not arg:
            arg = input("Resume session index or path: ").strip()
        if arg.isdigit():
            idx = int(arg)
            if idx <0 or idx >= len(files):
                print(f"Invalid session index: {idx}")
                return
            return resume_session(files[idx])
        return resume_session(arg)
    if cmd == "state":
        print(f"MODEL: {MODEL}")
        print(f"SESSION_FILE: {SESSION_FILE}")
        print(f"SESSION_ID: {SESSION_ID}")
        print(f"MESSAGES: {len(messages)}")
        print(f"TOOLS: {list(tool_impls.keys())}")
        return
    if cmd == "model":
        if not arg:
            print(f"Current model: {MODEL}")
            return MODEL
        MODEL = arg
        print(f"Model set to: {MODEL}")
        return MODEL

    if cmd == "tools":
        for name in tool_impls.keys():
            mark = "confirm" if name in TOOLS_REQUIRE_CONFIRM else "auto"
            print(f"- {name} ({mark})")
        return 
    print(f"Unknown command: /{cmd}")
    print("Run /help")
    

### use run as entry

In [474]:
def run(text, **kwargs):
    text = text.strip()

    if not text:
        return
    if text.startswith("/"):
        return handle_command(text)
    if SESSION_FILE is None:
        auto_resume_or_new()
    return ask(text, **kwargs)

### test command system

In [475]:
run("/help")

Available commands:
   /help              Show commands
   /new               Start a new session
   /sessions          List sessions
   /resume [index]    Resume session by index or path
   /state             Show current state
   /model [name]      Show or set model
   /tools             Show available tools


In [476]:
run("/state")

MODEL: deepseek-v4-flash
SESSION_FILE: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
SESSION_ID: b573371535a2456b83312d2d1a12f513
MESSAGES: 9
TOOLS: ['list_files', 'read_file', 'write_file', 'edit_file', 'bash']


In [477]:
run("/sessions")

[0] 20260730-143120-b5733715.jsonl
    messages: 8
    first: 请查看当前目录并简单总结
[1] 20260730-143120-90fe342b.jsonl
    messages: 0
    first: 
[2] 20260730-142022-fork-19228dcd.jsonl
    messages: 15
    first: 请查看当前目录并简单总结
[3] 20260730-141904-fork-dfb9b22e.jsonl
    messages: 13
    first: 请查看当前目录并简单总结
[4] 20260716-131755-8345e18d.jsonl
    messages: 198
    first: 请查看当前目录并简单总结
[5] 20260714-112646-7848e15f.jsonl
    messages: 14
    first: 请查看当前目录并简单总结
[6] 20260714-112553-9a6f27e6.jsonl
    messages: 3
    first: 请查看当前目录并简单总结
[7] 20260712-232412-054645d4.jsonl
    messages: 0
    first: 


[WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-b5733715.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-143120-90fe342b.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-142022-fork-19228dcd.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-141904-fork-dfb9b22e.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260716-131755-8345e18d.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260714-112646-7848e15f.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260714-112553-9a6f27e6.jsonl'),
 WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260712-232412-054645d4.jsonl')]

In [478]:
run("/resume 0")

[0] 20260730-143120-b5733715.jsonl
    messages: 8
    first: 请查看当前目录并简单总结
[1] 20260730-143120-90fe342b.jsonl
    messages: 0
    first: 
[2] 20260730-142022-fork-19228dcd.jsonl
    messages: 15
    first: 请查看当前目录并简单总结
[3] 20260730-141904-fork-dfb9b22e.jsonl
    messages: 13
    first: 请查看当前目录并简单总结
[4] 20260716-131755-8345e18d.jsonl
    messages: 198
    first: 请查看当前目录并简单总结
[5] 20260714-112646-7848e15f.jsonl
    messages: 14
    first: 请查看当前目录并简单总结
[6] 20260714-112553-9a6f27e6.jsonl
    messages: 3
    first: 请查看当前目录并简单总结
[7] 20260712-232412-054645d4.jsonl
    messages: 0
    first: 
Resumed session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
Restored messages: 9


In [479]:
run("/tools")

- list_files (auto)
- read_file (auto)
- write_file (confirm)
- edit_file (confirm)
- bash (confirm)


In [480]:
run("查看当前目录并总结")

**Tool call:** `list_files`

```text
.git/
.gitignore
.ipynb_checkpoints/
.mini-pi/
.nb-agent/
harness_demo.ipynb
harness_lab/
mini_harness.ipynb
notes/
README.txt
```

目录结构与之前一致，总结如下：

---

### 工作区概览：`agent_harness`

**核心理念**：学习 Pi agent 的 harness 框架，通过从零构建 Python 版本来实践。

| 文件/目录 | 说明 |
|-----------|------|
| `README.txt` | 项目描述文档 |
| `harness_demo.ipynb` | 主演示 Notebook |
| `mini_harness.ipynb` | 迷你版 harness Notebook |
| `harness_lab/` | 实验数据：BigCodeBench 基准测试的 JSON 配置、候选池、验证池及 CSV 结果，含 `runs/` 子目录 |
| `notes/` | 学习笔记：编辑演示、mini-pi 入门、权限测试，以及一篇**双皮奶做法** 🍮 |
| `.git/` + `.gitignore` | Git 管理 |
| `.ipynb_checkpoints/` | Jupyter 自动检查点 |
| `.mini-pi/` + `.nb-agent/` | 隐藏配置/状态目录 |

**一句话**：一个基于 Jupyter Notebook + BigCodeBench 评测数据、从零实现 agent harness 的动手学习项目。

{'content': '目录结构与之前一致，总结如下：\n\n---\n\n### 工作区概览：`agent_harness`\n\n**核心理念**：学习 Pi agent 的 harness 框架，通过从零构建 Python 版本来实践。\n\n| 文件/目录 | 说明 |\n|-----------|------|\n| `README.txt` | 项目描述文档 |\n| `harness_demo.ipynb` | 主演示 Notebook |\n| `mini_harness.ipynb` | 迷你版 harness Notebook |\n| `harness_lab/` | 实验数据：BigCodeBench 基准测试的 JSON 配置、候选池、验证池及 CSV 结果，含 `runs/` 子目录 |\n| `notes/` | 学习笔记：编辑演示、mini-pi 入门、权限测试，以及一篇**双皮奶做法** 🍮 |\n| `.git/` + `.gitignore` | Git 管理 |\n| `.ipynb_checkpoints/` | Jupyter 自动检查点 |\n| `.mini-pi/` + `.nb-agent/` | 隐藏配置/状态目录 |\n\n**一句话**：一个基于 Jupyter Notebook + BigCodeBench 评测数据、从零实现 agent harness 的动手学习项目。',
 'role': 'assistant',
 'reasoning_content': "Same result as before. I'll provide the summary directly."}

## context compression

### context statistic

In [481]:
def estimate_context_chars():
    return sum(
        len(json.dumps(m,ensure_ascii=False))
        for m in messages
    )
def context_stats():
    print(f"messages:{len(messages)}")
    print(f"estimated chars:{estimate_context_chars()}")
    print(f"session:{SESSION_FILE}")

In [482]:
context_stats()

messages:13
estimated chars:4642
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


### messages to text

In [483]:
def shorten(text, max_chars=3000):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars]+f"\n...[truncated {len(text)} chars]"

def format_message_for_summary(m):
    role = m.get("role")

    if role == "user":
        return f"USER:\n{shorten(m.get('content',''))}"
    if role == "assistant":
        parts = []

        content = m.get("content")
        if content:
            parts.append(f"ASSISTANT:\n{shorten(content)}")
        tool_calls = m.get("tool_calls") or []
        if tool_calls:
            parts.append(
                "ASSISTANT TOOL CALLS:\n"
                + shorten(json.dumps(tool_calls, ensure_ascii=False, indent=2))
            )
        return "\n\n".join(parts) or "ASSISTANT:"
    if role == "tool":
        return f"TOOL RESULT:\n{shorten(m.get('content',''))}"
    return f"{role}:\n{shorten(m)}"

def transcript_for_summary(msgs, max_chars=60000):
    text = "\n\n---\n\n".join(format_message_for_summary(m) for m in msgs)

    if len(text)<=max_chars:
        return text

    half = max_chars // 2
    return (
           text[:half]
           + "\n\n...[middle omitted during compaction]...\n\n"
           + text[-half:]
       )

### use model to generate summary

In [484]:
def summarize_messages(msgs):
    transcript = transcript_for_summary(msgs)

    prompt = f"""
    Summarize this agent conversation for future continuation.
    
    Preserve:
    - user goals and constraints
    - important decisions
    - files inspected
    - files created or modified
    - commands run and important results
    - current task state
    - anything the future assistant must remember
    
    Be concise but complete.
    
    Transcript:

    {transcript}
    """.strip()
    resp = client.chat.completions.create(
        model = MODEL,
        messages=[
            {
                "role":"system",
                "content": "You summarize coding-agent conversations accurately.",
            },
            {
                "role":"user",
                "content":prompt,
            },
        ]
    )
    return resp.choices[0].message.content


### construct the compressed summary message

In [485]:
def make_compaction_message(summary):
    return {
       "role": "user",
       "content": f"""
    The conversation before this point was compacted.
    
    Use this summary as prior context:
    
    <summary>
    {summary}
    </summary>
    """.strip()
    }

### chose the compress point

In [486]:
# 从最近若干消息往前找一个user消息作为切点
def choose_compaction_cut(keep_recent=12):
    if len(messages) <= keep_recent +2:
        return None
    cut = max(1, len(messages)-keep_recent)
    while cut >1 and messages[cut].get("role") != "user":
        cut -= 1
    if cut <= 1:
        return None
    return cut

### realize compact_context

In [487]:
def compact_context(keep_recent=12):
    global messages

    if SESSION_FILE is None:
        raise RuntimeError("No active session. Run new_session() or auto_resume_or_new()")
    cut = choose_compaction_cut(keep_recent)

    if cut is None:
        print("Nothing to compact")
        return None

    system = messages[0]
    older = messages[1:cut]
    recent = messages[cut:]
    print(f"Compacting {len(older)} older messages...")
    print(f"Keeping {len(recent)} recent messages...")

    summary = summarize_messages(older)
    summary_msg = make_compaction_message(summary)

    append_session_entry({
        "type":"compaction",
        "summary":summary,
        "olderMessageCount":len(older),
        "keptMessages":recent,
    })
    messages = [system, summary_msg] + recent

    print("Compaction complete")
    context_stats()
    return summary
    

In [488]:
# compact_context(keep_recent = 4)
messages

[{'role': 'system',
  'content': 'You are a minimal coding agent running inside Jupyter Notebook.\n\nWorkspace root:\nD:\\workflow\\agent_harness\n\nAvailable tools:\n- list_files: list files in a directory.\n- read_file: read a text file.\n- write_file: create or overwrite a text file.\n- edit_file: edit a file by exact text replacement.\n- bash: run shell commands in the workspace.\n\nRules:\n- Use list_files/read_file before modifying existing files.\n- Use edit_file for targeted modifications.\n- oldText in edit_file must match exactly once.\n- Use write_file only when creating a new file or replacing a full file.\n- Use bash for inspection, tests, search, and simple scripts.\n- Avoid destructive commands.\n- Be concise.'},
 {'role': 'user', 'content': '请查看当前目录并简单总结'},
 {'content': '',
  'role': 'assistant',
  'tool_calls': [{'id': 'call_00_GVKVybirqVJLuA4qmzoX2327',
    'function': {'arguments': '{"path": "."}', 'name': 'list_files'},
    'type': 'function',
    'index': 0}],
  'r

### revise resume_session

In [565]:
# identify the new compaction, replace resume_session by the following
def resume_session(path):
    global SESSION_FILE, SESSION_ID, SESSION_LEAF_ID, messages

    SESSION_FILE = Path(path)
    entries = read_jsonl(SESSION_FILE)

    if not entries or entries[0].get("type") != "session":
        raise ValueError("Not a valid mini-pi session file")

    SESSION_ID = entries[0]["id"]

    SESSION_LEAF_ID = None
    for e in entries[1:]:
        if "id" in e:
            SESSION_LEAF_ID = e["id"]

    system = current_system_message()

    latest_compaction_idx = None
    for i, e in enumerate(entries):
        if e.get("type") == "compaction":
            latest_compaction_idx = i
    restored = [system]

    if latest_compaction_idx is not None:
        comp = entries[latest_compaction_idx]
        restored.append(make_compaction_message(comp["summary"]))

        restored.extend(comp.get("keptMessages",[]))

        source_entries = entries[latest_compaction_idx:1]
    else:
        source_entries = entries[1:]

    for e in source_entries:
        if e.get("type") == "message":
            restored.append(e["message"])

    messages = restored

    print(f"Resumed session: {SESSION_FILE}")
    print(f"Restored message: {len(messages)}")
    context_stats()
    



In [490]:
resume_session(SESSION_FILE)

Resumed session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl
Restored message: 13
messages:13
estimated chars:4642
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


In [491]:
len(messages)

13

### /compact,/context in command system

In [492]:
# use wrapper instead of rewrite handle_command
if "_base_handle_command_without_compaction" not in globals():
    _base_handle_command_without_compaction = handle_command

def handle_command(text):
    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_compaction(text)
        print("""
        Extra commands:
        /context                 Show current context size
        /compact [keep_recent]   Compact old messages, keeping recent N
        messages
        """.strip())
        return result
    if cmd == "context":
        return context_stats()

    if cmd == "compact":
        keep_recent = int(arg) if arg else 12
        return compact_context(keep_recent = keep_recent)
    return _base_handle_command_without_compaction

In [493]:
run("/context")

messages:13
estimated chars:4642
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


In [494]:
run("/compact 3")

Compacting 8 older messages...
Keeping 4 recent messages...
Compaction complete
messages:6
estimated chars:3044
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


'## 对话总结\n\n**用户目标**：查看当前目录并获取概要。\n\n**文件查看**：\n- 读取 `README.txt`（项目描述：学习 Pi agent 的 harness 框架，从简单到复杂构建 Python 版本）\n- 列出根目录文件和 `harness_lab/`、`notes/` 子目录内容\n\n**目录结构（Agent Harness 项目）**：\n- 根目录：`harness_demo.ipynb`、`mini_harness.ipynb`、`harness_lab/`（含 BigCodeBench 评测数据与结果）、`notes/`、`.git/`、`.gitignore`、`.ipynb_checkpoints/`、`.mini-pi/`、`.nb-agent/`\n- `harness_lab/`：基准 JSON 配置（benchmark_bigcodebench_sample.json, candidate pool, valid pool）、CSV 验证结果、runs/ 目录\n- `notes/`：编辑 demo、mini-pi、权限测试笔记及一个食谱笔记\n\n**当前任务状态**：已完成用户请求，用户获得目录概要。项目处于早期探索阶段，无代码修改或命令执行（仅读操作）。\n\n**未来继续需记住**：项目目标是实现类 Pi agent harness，使用 Jupyter Notebook 原型，附带 BigCodeBench 评测实验数据。可在此基础上扩展 harness 功能或运行实验。'

In [495]:
run("git commit当前项目")

**Tool call:** `bash`


Tool `bash` wants to run with args:
{
  "command": "cd D:\\workflow\\agent_harness && git status"
}


Allow? [y/N]  y


```text
$ cd D:\workflow\agent_harness && git status
       exit_code: 0

       STDOUT:
       On branch master
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   mini_harness.ipynb
	modified:   notes/edit-demo.md

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notes/permission-test.md

no changes added to commit (use "git add" and/or "git commit -a")


       STDERR:
       
       
```

有变更：`mini_harness.ipynb` 和 `notes/edit-demo.md` 修改过，新增 `notes/permission-test.md`。看一下改了什么内容，然后一次性提交。

...


Tool `bash` wants to run with args:
{
  "command": "cd D:\\workflow\\agent_harness && git add -A && git commit -m \"feat: permission-test note & line-ending normalization\n\n- Add notes/permission-test.md (hello permission)\n- Normalize line endings in mini_harness.ipynb and notes/edit-demo.md\""
}


Allow? [y/N]  y


```text
$ cd D:\workflow\agent_harness && git add -A && git commit -m "feat: permission-test note & line-ending normalization

- Add notes/permission-test.md (hello permission)
- Normalize line endings in mini_harness.ipynb and notes/edit-demo.md"
       exit_code: 0

       STDOUT:
       [master 823f898] feat: permission-test note & line-ending normalization
 3 files changed, 6746 insertions(+), 248 deletions(-)
 create mode 100644 notes/permission-test.md


       STDERR:
       warning: in the working copy of 'mini_harness.ipynb', LF will be replaced by CRLF the next time Git touches it

       
```

✅ 提交成功！Commit `823f898` 包含 3 个文件变更：

| 文件 | 变更 |
|------|------|
| `mini_harness.ipynb` | 6,745 行变动（行尾规范统一） |
| `notes/edit-demo.md` | 行尾规范统一 |
| `notes/permission-test.md` | **新增**（内容：`hello permission`） |

提交信息：*"feat: permission-test note & line-ending normalization"*。工作区现已干净。

{'content': '✅ 提交成功！Commit `823f898` 包含 3 个文件变更：\n\n| 文件 | 变更 |\n|------|------|\n| `mini_harness.ipynb` | 6,745 行变动（行尾规范统一） |\n| `notes/edit-demo.md` | 行尾规范统一 |\n| `notes/permission-test.md` | **新增**（内容：`hello permission`） |\n\n提交信息：*"feat: permission-test note & line-ending normalization"*。工作区现已干净。',
 'role': 'assistant',
 'reasoning_content': 'Commit was successful.'}

## auto compaction

run("")->ask()->check the length of current context->if too long, compact_context()->continue

### compaction configure

In [ ]:
AUTO_COMPACT_ENABLED = True
AUTO_COMPACT_CHAR_LIMIT = 80_000
AUTO_COMPACT_KEEP_RECENT = 12

### add status function

In [497]:
def auto_compact_status():
   print(f"AUTO_COMPACT_ENABLED: {AUTO_COMPACT_ENABLED}")
   print(f"AUTO_COMPACT_CHAR_LIMIT: {AUTO_COMPACT_CHAR_LIMIT}")
   print(f"AUTO_COMPACT_KEEP_RECENT: {AUTO_COMPACT_KEEP_RECENT}")
   context_stats()

In [498]:
auto_compact_status()

AUTO_COMPACT_ENABLED: True
AUTO_COMPACT_CHAR_LIMIT: 80000
AUTO_COMPACT_KEEP_RECENT: 12
messages:21
estimated chars:9809
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


### realize the check for auto compression

In [499]:
def maybe_auto_compact(reason="threshold"):
    if not AUTO_COMPACT_ENABLED:
        return None
    
    if SESSION_FILE is None:
        return None
    
    chars = estimate_context_chars()
    
    if chars < AUTO_COMPACT_CHAR_LIMIT:
        return None
    
    print(f"[auto compact] reason={reason}")
    print(f"[auto compact] context chars={chars},limit={AUTO_COMPACT_CHAR_LIMIT}")
    
    return compact_context(keep_recent=AUTO_COMPACT_KEEP_RECENT)

### replace ask function

In [500]:
# 每次call_model前都会检查是否需要自动压缩
def ask(prompt,max_turns=50):
    if SESSION_FILE is None:
        raise RuntimeError("No active session. Run new_session() or resume_session(path) first.")

    user_msg = {
        "role": "user",
        "content": prompt,
       }
    messages.append(user_msg)
    save_message(user_msg)

    for turn in range(max_turns):
        maybe_auto_compact(reason = "before_model_call")
        assistant_msg = call_model()

        messages.append(assistant_msg)
        save_message(assistant_msg)
        content = assistant_msg.get("content")
        if content:
            display(Markdown(content))

        tool_calls = assistant_msg.get("tool_calls") or []

        if not tool_calls:
            return assistant_msg
        for tool_call in tool_calls:
            name = tool_call["function"]["name"]
            
            display(Markdown(f"**Tool call:** `{name}`"))
            
            result = execute_tool_call(tool_call)
            
            display(Markdown(f"```text\n{result[:3000]}\n```"))
            
            tool_msg = {
                
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": result,
            }
            
            messages.append(tool_msg)
            save_message(tool_msg)
    raise RuntimeError("Agent exceeded max_turns")

### add /auto-compact in system command

In [501]:
if "_base_handle_command_without_auto_compact" not in globals():
    _base_handle_command_without_auto_compact = handle_command

def handle_command(text):
    global AUTO_COMPACT_ENABLED
    global AUTO_COMPACT_CHAR_LIMIT
    global AUTO_COMPACT_KEEP_RECENT
    
    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""
    
    if cmd in ("help", "?"):
        result = _base_handle_command_without_auto_compact(text)
        print("""
            Auto compaction commands:
            /auto-compact                 Show auto compaction status
            /auto-compact on              Enable auto compaction
            /auto-compact off             Disable auto compaction
            /auto-compact limit N         Set char limit
            /auto-compact keep N          Set recent messages to keep
            """.strip())
        return result

    if cmd == "auto-compact":
        if not arg:
            return auto_compact_status()
    
        if arg == "on":
            AUTO_COMPACT_ENABLED = True
            print("Auto compaction enabled.")
            return
    
        if arg == "off":
            AUTO_COMPACT_ENABLED = False
            print("Auto compaction disabled.")
            return
    
        sub = arg.split(maxsplit=1)
        if len(sub) == 2 and sub[0] == "limit":
            AUTO_COMPACT_CHAR_LIMIT = int(sub[1])
            print(f"Auto compaction char limit set to {AUTO_COMPACT_CHAR_LIMIT}")
            return
    
        if len(sub) == 2 and sub[0] == "keep":
            AUTO_COMPACT_KEEP_RECENT = int(sub[1])
            print(f"Auto compaction keep_recent set to {AUTO_COMPACT_KEEP_RECENT}")
            return
    
        print("Usage: /auto-compact [on|off|limit N|keep N]")
        return
    
    return _base_handle_command_without_auto_compact(text)

### 测试自动 compaction

In [502]:
run("/auto-compact")

AUTO_COMPACT_ENABLED: True
AUTO_COMPACT_CHAR_LIMIT: 80000
AUTO_COMPACT_KEEP_RECENT: 12
messages:21
estimated chars:9809
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


In [503]:
run("/auto-compact limit 5000")

Auto compaction char limit set to 5000


In [504]:
run("请查看当前目录并总结")

[auto compact] reason=before_model_call
[auto compact] context chars=9850,limit=5000
Compacting 5 older messages...
Keeping 16 recent messages...
Compaction complete
messages:18
estimated chars:8209
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


**Tool call:** `list_files`

```text
.git/
.gitignore
.ipynb_checkpoints/
.mini-pi/
.nb-agent/
harness_demo.ipynb
harness_lab/
mini_harness.ipynb
notes/
README.txt
```

[auto compact] reason=before_model_call
[auto compact] context chars=8720,limit=5000
Compacting 1 older messages...
Keeping 18 recent messages...
Compaction complete
messages:20
estimated chars:8577
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


**Tool call:** `list_files`

...

**Tool call:** `read_file`

```text
1: 本项目学习Pi agent的harness框架，从简单到复杂，构建一个类似的python版本，通过实践来学习agent harness
```

[auto compact] reason=before_model_call
[auto compact] context chars=9715,limit=5000
Compacting 1 older messages...
Keeping 22 recent messages...
Compaction complete
messages:24
estimated chars:9709
session:D:\workflow\agent_harness\.mini-pi\sessions\20260730-143120-b5733715.jsonl


---

## 项目概览

**项目**：`agent_harness` — 学习 Pi agent 的 harness 框架，通过实践构建类似 Python 版本。

### 目录结构

```
agent_harness/
├── README.txt                  # 项目说明
├── .gitignore
├── .ipynb-checkpoints/         # 自动生成的 notebook 缓存
├── .mini-pi/                   # mini-pi 运行时目录
├── .nb-agent/                  # nb-agent 配置/状态目录
│
├── harness_demo.ipynb          # Pi agent harness 演示 notebook
├── mini_harness.ipynb          # 自建 mini harness notebook（核心实验文件）
│
├── notes/                      # 笔记与文档
│   ├── edit-demo.md            # 编辑 demo 笔记
│   ├── hello-mini-pi.md        # mini-pi 入门笔记
│   ├── permission-test.md      # 权限测试笔记（最新新增）
│   └── 双皮奶做法.md           # 配方笔记（无代码相关）
│
└── harness_lab/                # 实验数据
    ├── benchmark_bigcodebench_sample.json
    ├── bigcodebench_hard_candidate_pool.json
    ├── bigcodebench_hard_valid_pool.json
    ├── canonical_validation_results.csv
    └── runs/                   # 实验运行记录
```

### Git 状态

- **当前分支**：`master`
- **最新提交**：`823f898` — *feat: permission-test note & line-ending normalization*
- ✅ 工作区干净，无未提交更改

### 项目阶段

处于**早期探索实验阶段**，核心是迭代 `mini_harness.ipynb` 学习 Pi agent 框架，并利用 `harness_lab/` 中的 BigCodeBench 数据进行基准测试实验。

{'content': '---\n\n## 项目概览\n\n**项目**：`agent_harness` — 学习 Pi agent 的 harness 框架，通过实践构建类似 Python 版本。\n\n### 目录结构\n\n```\nagent_harness/\n├── README.txt                  # 项目说明\n├── .gitignore\n├── .ipynb-checkpoints/         # 自动生成的 notebook 缓存\n├── .mini-pi/                   # mini-pi 运行时目录\n├── .nb-agent/                  # nb-agent 配置/状态目录\n│\n├── harness_demo.ipynb          # Pi agent harness 演示 notebook\n├── mini_harness.ipynb          # 自建 mini harness notebook（核心实验文件）\n│\n├── notes/                      # 笔记与文档\n│   ├── edit-demo.md            # 编辑 demo 笔记\n│   ├── hello-mini-pi.md        # mini-pi 入门笔记\n│   ├── permission-test.md      # 权限测试笔记（最新新增）\n│   └── 双皮奶做法.md           # 配方笔记（无代码相关）\n│\n└── harness_lab/                # 实验数据\n    ├── benchmark_bigcodebench_sample.json\n    ├── bigcodebench_hard_candidate_pool.json\n    ├── bigcodebench_hard_valid_pool.json\n    ├── canonical_validation_results.csv\n    └── runs/                   # 实验运行记录\n```\n\n### Git 状态\n\n- **当前分支

In [505]:
run("/auto-compact limit 800000")

Auto compaction char limit set to 800000


## 增加hooks

### 定义 hooks 注册表 与 注册hooks

In [506]:
TOOL_HOOKS = {
    "tool_call":[],
    "tool_result":[],
}

def register_tool_hook(event_type, fn, name = None):
    if event_type not in TOOL_HOOKS:
        raise ValueError(f"Unknow hook event type: {event_type}")

    hook = {
    "name": name or fn.__name__,
    "fn":fn,
    "enabled":True,
    }
    TOOL_HOOKS[event_type].append(hook)
    return hook

### 查看/开关 hook

In [507]:
def list_tool_hooks():
    for event_type, hooks in TOOL_HOOKS.items():
        print(f"{event_type}:")
        if not hooks:
            print(" <none>")
        for h in hooks:
            status = "on" if h["enabled"] else "off"
            print(f" -{h['name']} [{status}]")

def set_tool_hook_enabled(name, enabled):
    found = False

    for hooks in TOOL_HOOKS.values():
        for h in hooks:
            if h["name"] == name:
                h["enabled"] = enabled
                found = True
                print(f"{name}: {'on' if enabled else 'off'}")

In [508]:
list_tool_hooks()

tool_call:
 <none>
tool_result:
 <none>


### 执行hooks

In [509]:
# tool_call hook 支持：

#  ```python
#    return {"block": True, "reason": "..."}
#    return {"args": new_args}
#  ```

#  tool_result hook 支持：

#  ```python
#    return {"result": new_result}
#    return {"is_error": True}

In [510]:
def emit_tool_call_hooks(event):
    for h in TOOL_HOOKS["tool_call"]:
        if not h["enabled"]:
            continue
        try:
            result = h["fn"](event)
        except Exception as e:
            return {
                "block": True,
                "reason": f"tool_call hook {h['name']} failed: {type(e).__name__}: {e}",
            }
        if not result:
            continue
        if result.get("block"):
            return {
                "block": True,
                "reason": result.get("reason", f"blocked by hook {h['name']}"),
            }
        if "args" in result:
            event["args"] = result["args"]

    return {"block": False, "event": event}

def emit_tool_result_hooks(event):
    for h in TOOL_HOOKS["tool_result"]:
        if not h["enabled"]:
            continue
        try:
            result = h["fn"](event)
        except Exception as e:
            print(f"[hook warning] tool_result hook {h['name']} failed: {e}")
            continue
        if not result:
            continue

        if "result" in result:
            event["result"] = result["result"]
        if "is_error" in result:
            event["is_error"] = result["is_error"]

    return event

### 替换 execute_tool_call

In [511]:
def execute_tool_call(tool_call):
    name = tool_call["function"]["name"]
    raw_args = tool_call["function"].get("arguments") or "{}"

    try:
        args = json.loads(raw_args)
    except Exception as e:
        return f"ERROR: failed to parse tool arguments: {e}"
    fn = tool_impls.get(name)
    if fn is None:
        return f"ERROR: unknown tool: {name}"
    call_event = {
    "type": "tool_call",
    "tool": name,
    "args": args,
    "rawArgs": raw_args,
    "timestamp": time.time(),
    }

    hook_result = emit_tool_call_hooks(call_event)
    if hook_result.get("block"):
        return f"BLOCKED: {hook_result.get('reason')}"
    args = hook_result["event"]["args"]

    if name in TOOLS_REQUIRE_CONFIRM:
        allowed = confirm_tool_call(name,args)
        if not allowed:
            return "BLOCKED: user rejected tool call"

    started = time.time()
    is_error = False

    try:
        result = fn(**args)
    except Exception as e:
        is_error = True
        result = f"ERROR: {type(e).__name__}: {e}"

    duration_ms = int((time.time()-started)*1000)

    result_event = {
        "type": "tool_result",
        "tool": name,
        "args": args,
        "result": str(result),
        "is_error": is_error,
        "duration_ms": duration_ms,
        "timestamp": time.time(),
    }
    result_event = emit_tool_result_hooks(result_event)
    return result_event["result"]
    
            

### 几个实用的hooks

In [512]:
#1. 工具调用日志
TOOL_CALL_LOG = []

def log_tool_call(event):
    TOOL_CALL_LOG.append({
        "tool": event["tool"],
        "args": event["args"],
        "timestamp": event["timestamp"],
    })

#注册
register_tool_hook("tool_call",log_tool_call,name = "log_tool_call")

{'name': 'log_tool_call',
 'fn': <function __main__.log_tool_call(event)>,
 'enabled': True}

In [513]:
#2. 保护敏感路径
# 阻止读写这些路径：

 # ```text
 #   .env
 #   .git/
 #   .mini-pi/
 # ```

PROTECTED_PATH_PARTS = {
   ".git",
   ".mini-pi",
}

PROTECTED_FILE_NAMES = {
   ".env",
   ".env.local",
   ".env.production",
}
def extract_paths_from_tool_call(tool, args):
    if tool in ("list_files", "read_file", "write_file","edit_file"):
        path = args.get("path")
        return [path] if path else []

    return []

def protect_sensitive_paths(event):
    tool = event["tool"]
    args = event["args"]

    for path in extract_paths_from_tool_call(tool, args):
        try:
            p = resolve_workspace_path(path)
        except Exception as e:
            return {
               "block": True,
               "reason": str(e),
            }

        parts = set(p.parts)

        if p.name in PROTECTED_FILE_NAMES:
            return {
               "block": True,
               "reason": f"protected file: {path}",
            }

        if parts & PROTECTED_PATH_PARTS:
            return {
               "block": True,
               "reason": f"protected path: {path}",
            }

# 注册
register_tool_hook("tool_call", protect_sensitive_paths,name="protect_sensitive_paths")

{'name': 'protect_sensitive_paths',
 'fn': <function __main__.protect_sensitive_paths(event)>,
 'enabled': True}

In [514]:
# 3. 限制 read_file 的最大行数（防止模型一次读取太大文件）
def cap_read_file_limit(event):
    if event["tool"] != "read_file":
        return
    
    args = dict(event["args"])
    limit = int(args.get("limit", 200))
    
    if limit > 500:
        args["limit"] = 500
        return {"args": args}

# 注册
register_tool_hook("tool_call", cap_read_file_limit, name="cap_read_file_limit")

{'name': 'cap_read_file_limit',
 'fn': <function __main__.cap_read_file_limit(event)>,
 'enabled': True}

In [515]:
# 4. 阻止危险 bash
def block_dangerous_bash(event):
    if event["tool"] != "bash":
        return
    
    command = event["args"].get("command", "")
    
    blocked_patterns = [
       "sudo ",
       "rm -rf",
       "chmod -R 777",
       "chown -R",
       "mkfs",
       "dd if=",
       ":(){",
    ]
    
    for pattern in blocked_patterns:
        if pattern in command:
            return {
               "block": True,
               "reason": f"dangerous bash pattern: {pattern}",
            }

# 注册
register_tool_hook("tool_call", block_dangerous_bash, name="block_dangerous_bash")

{'name': 'block_dangerous_bash',
 'fn': <function __main__.block_dangerous_bash(event)>,
 'enabled': True}

In [516]:
# 5. 工具结果脱敏（疑似 key/token 简单打码）

import re

SECRET_PATTERNS = [
   r"sk-[A-Za-z0-9_\-]{20,}",
   r"ghp_[A-Za-z0-9_]{20,}",
   r"AIza[0-9A-Za-z_\-]{20,}",
]

def redact_secrets_in_tool_result(event):
    text = event["result"]
    
    for pattern in SECRET_PATTERNS:
        text = re.sub(pattern, "[REDACTED_SECRET]", text)
    
    if text != event["result"]:
        return {"result": text}

# 注册
register_tool_hook("tool_result", redact_secrets_in_tool_result, name="redact_secrets")

{'name': 'redact_secrets',
 'fn': <function __main__.redact_secrets_in_tool_result(event)>,
 'enabled': True}

### 测试 hooks

In [517]:
# 查看 hooks
list_tool_hooks()

tool_call:
 -log_tool_call [on]
 -protect_sensitive_paths [on]
 -cap_read_file_limit [on]
 -block_dangerous_bash [on]
tool_result:
 -redact_secrets [on]


In [518]:
# 测试保护路径
fake_call = {
   "id": "test",
   "function": {
       "name": "read_file",
       "arguments": json.dumps({"path": ".env"}),
   },
}
execute_tool_call(fake_call)

'BLOCKED: protected file: .env'

In [519]:
# 测试 bash 阻止
fake_call = {
       "id": "test",
       "function": {
           "name": "bash",
           "arguments": json.dumps({"command": "sudo ls"}),
       },
   }
execute_tool_call(fake_call)

'BLOCKED: dangerous bash pattern: sudo '

### 增加 /hooks 命令

In [520]:
if "_base_handle_command_without_hooks" not in globals():
    _base_handle_command_without_hooks = handle_command

def handle_command(text):
    parts = text.strip().split(maxsplit=2)
    cmd = parts[0][1:]
    arg1 = parts[1].strip() if len(parts) > 1 else ""
    arg2 = parts[2].strip() if len(parts) > 2 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_hooks(text)
        print("""
        Hook commands:
        /hooks                 List tool hooks
        /hooks on NAME         Enable hook
        /hooks off NAME        Disable hook
        """.strip())
        return result

    if cmd == "hooks":
        if not arg1:
            return list_tool_hooks()
    
        if arg1 == "on" and arg2:
            return set_tool_hook_enabled(arg2, True)
    
        if arg1 == "off" and arg2:
            return set_tool_hook_enabled(arg2, False)
    
        print("Usage: /hooks [on|off NAME]")
        return
    return _base_handle_command_without_hooks(text)

In [521]:
# 测试
run("/hooks")

tool_call:
 -log_tool_call [on]
 -protect_sensitive_paths [on]
 -cap_read_file_limit [on]
 -block_dangerous_bash [on]
tool_result:
 -redact_secrets [on]


In [522]:
run("/hooks off cap_read_file_limit")

cap_read_file_limit: off


In [523]:
run("/hooks on cap_read_file_limit")

cap_read_file_limit: on


## 增加权限策略系统

### 定义权限动作+session级允许列表

In [524]:
PERMISSION_ALLOW = "allow"
PERMISSION_CONFIRM = "confirm"
PERMISSION_BLOCK = "block"

SESSION_ALLOWED_TOOL_PATTERNS = set()

### 工具风险评估函数

In [525]:
def permission_key_for_tool(tool, args):
    if tool in ("list_files", "read_file"):
        return f"{tool}:readonly"
    
    if tool in ("write_file", "edit_file"):
        return f"{tool}:{args.get('path', '')}"
    
    if tool == "bash":
        command = args.get("command", "").strip()
        first_word = command.split()[0] if command.split() else ""
        return f"bash:{first_word}"
        
    return f"{tool}:default"

In [526]:
def assess_tool_permission(tool, args):
    key = permission_key_for_tool(tool, args)

    if key in SESSION_ALLOWED_TOOL_PATTERNS:
        return {
           "action": PERMISSION_ALLOW,
           "reason": f"allowed for session: {key}",
           "key": key,
        }

   # 只读默认允许
    if tool in ("list_files", "read_file"):
        return {
           "action": PERMISSION_ALLOW,
           "reason": "read-only tool",
           "key": key,
        }

   # 写文件需要确认
    if tool in ("write_file", "edit_file"):
        return {
           "action": PERMISSION_CONFIRM,
           "reason": "file modification",
           "key": key,
        }

   # bash 风险判断
    if tool == "bash":
        command = args.get("command", "")

        blocked = [
        "rm -rf",
        "sudo",
        "mkfs",
        "dd if=",
        "chmod -R 777",
        "chown -R",
        ":(){",
        ]

        for pattern in blocked:
            if pattern in command:
                return {
                "action": PERMISSION_BLOCK,
                "reason": f"blocked dangerous bash pattern: {pattern}",
                "key": key,
                }

        safe_prefixes = [
           "ls",
           "pwd",
           "find",
           "grep",
           "rg",
           "cat",
           "head",
           "tail",
           "python",
           "pytest",
           "git",
        ]

        first_word = command.strip().split()[0] if command.strip().split() else ""

        if first_word in safe_prefixes:
            return {
               "action": PERMISSION_CONFIRM,
               "reason": f"bash command: {first_word}",
               "key": key,
            }

        return {
           "action": PERMISSION_CONFIRM,
           "reason": "unknown bash command",
           "key": key,
        }

    return {
       "action": PERMISSION_CONFIRM,
       "reason": "unknown tool risk",
       "key": key,
    }

### 新确认函数： 一次允许 / 本 session 允许 / 拒绝

In [527]:
def confirm_tool_permission(tool, args, decision):
    print(f"\nTool `{tool}` requests permission:")
    print(f"Reason: {decision['reason']}")
    print(f"Permission key: {decision['key']}")
    print("Args:")
    print(json.dumps(args, ensure_ascii=False, indent=2))
    
    print("""
    Choose:
    y  = allow once
    s  = allow for this session
    n  = deny
    """.strip())

    answer = input("Permission? [y/s/N] ").strip().lower()
    
    if answer in ("y", "yes"):
       return True

    if answer in ("s", "session"):
       SESSION_ALLOWED_TOOL_PATTERNS.add(decision["key"])
       print(f"Allowed for session: {decision['key']}")
       return True

    return False

### 替换 execute_tool_call

In [528]:
def execute_tool_call(tool_call):
    name = tool_call["function"]["name"]
    raw_args = tool_call["function"].get("arguments") or "{}"

    try:
        args = json.loads(raw_args)
    except Exception as e:
        return f"ERROR: failed to parse tool arguments: {e}"

    fn = tool_impls.get(name)
    if fn is None:
        return f"ERROR: unknown tool: {name}"

    # 1. tool_call hooks
    call_event = {
       "type": "tool_call",
       "tool": name,
       "args": args,
       "rawArgs": raw_args,
       "timestamp": time.time(),
    }

    hook_result = emit_tool_call_hooks(call_event)

    if hook_result.get("block"):
        return f"BLOCKED: {hook_result.get('reason')}"

    args = hook_result["event"]["args"]

    # 2. permission policy
    decision = assess_tool_permission(name, args)

    if decision["action"] == PERMISSION_BLOCK:
        return f"BLOCKED: {decision['reason']}"

    if decision["action"] == PERMISSION_CONFIRM:
        allowed = confirm_tool_permission(name, args, decision)
        if not allowed:
            return "BLOCKED: user denied permission"

    # 3. execute
    started = time.time()
    is_error = False

    try:
        result = fn(**args)
    except Exception as e:
        is_error = True
        result = f"ERROR: {type(e).__name__}: {e}"

    duration_ms = int((time.time() - started) * 1000)

    # 4. tool_result hooks
    result_event = {
       "type": "tool_result",
       "tool": name,
       "args": args,
       "result": str(result),
       "is_error": is_error,
       "duration_ms": duration_ms,
       "timestamp": time.time(),
    }

    result_event = emit_tool_result_hooks(result_event)

    return result_event["result"]

### 增加权限管理命令

In [529]:
if "_base_handle_command_without_permissions" not in globals():
    _base_handle_command_without_permissions = handle_command


def list_session_permissions():
    if not SESSION_ALLOWED_TOOL_PATTERNS:
        print("No session permissions.")
        return

    print("Session permissions:")
    for item in sorted(SESSION_ALLOWED_TOOL_PATTERNS):
        print(f"- {item}")


def clear_session_permissions():
    SESSION_ALLOWED_TOOL_PATTERNS.clear()
    print("Cleared session permissions.")

In [530]:
def handle_command(text):
    parts = text.strip().split(maxsplit=2)
    cmd = parts[0][1:]
    arg1 = parts[1].strip() if len(parts) > 1 else ""
    arg2 = parts[2].strip() if len(parts) > 2 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_permissions(text)
        print("""
        Permission commands:
        /permissions              List session permissions
        /permissions clear        Clear session permissions
        """.strip())
        return result

    if cmd == "permissions":
        if arg1 == "clear":
           return clear_session_permissions()
        return list_session_permissions()

    return _base_handle_command_without_permissions(text)

### 测试

In [531]:
fake_call = {
   "id": "test",
   "function": {
       "name": "list_files",
       "arguments": json.dumps({"path": "."}),
   },
}
execute_tool_call(fake_call)

'.git/\n.gitignore\n.ipynb_checkpoints/\n.mini-pi/\n.nb-agent/\nharness_demo.ipynb\nharness_lab/\nmini_harness.ipynb\nnotes/\nREADME.txt'

In [532]:
# 写文件应确认


fake_call = {
   "id": "test",
   "function": {
       "name": "write_file",
       "arguments": json.dumps({
           "path": "notes/permission-test.md",
           "content": "hello permission"
       }, ensure_ascii=False),
   },
}

execute_tool_call(fake_call)


Tool `write_file` requests permission:
Reason: file modification
Permission key: write_file:notes/permission-test.md
Args:
{
  "path": "notes/permission-test.md",
  "content": "hello permission"
}
Choose:
    y  = allow once
    s  = allow for this session
    n  = deny


Permission? [y/s/N]  s


Allowed for session: write_file:notes/permission-test.md


'Wrote file: notes/permission-test.md (16 chars)'

In [533]:
# bash 按照命令类型确认
fake_call = {
   "id": "test",
   "function": {
       "name": "bash",
       "arguments": json.dumps({
           "command": "ls -la",
           "timeout": 10,
       }),
   },
}

execute_tool_call(fake_call)


Tool `bash` requests permission:
Reason: bash command: ls
Permission key: bash:ls
Args:
{
  "command": "ls -la",
  "timeout": 10
}
Choose:
    y  = allow once
    s  = allow for this session
    n  = deny


Permission? [y/s/N]  s


Allowed for session: bash:ls


"$ ls -la\n       exit_code: 1\n\n       STDOUT:\n       \n\n       STDERR:\n       'ls' 不是内部或外部命令，也不是可运行的程序\n或批处理文件。\n\n       "

## 模型、thinking 管理以及模型提供商

In [ ]:
MODEL = globals().get("MODEL","deepseek-v4-pro")
THINKING_LEVEL = globals().get("THINKING_LEVEL", "off")

THINKING_LEVELS = ["off", "minimal", "low", "medium", "high"]
MODEL_REGISTRY = {
   "deepseek-v4-flash": {
       "reasoning": False,
       "notes": "Cheaper/faster model"
   },
   "deepseek-v4-pro": {
       "reasoning": True,
       "notes": "Reasoning-capable model"
   },
}


### 模型能力判断

In [ ]:
def get_model_info(model=None):
    model = model or MODEL
    
    if model in MODEL_REGISTRY:
       return MODEL_REGISTRY[model]

    # 简单启发式：
    reasoning
    reasoning = (
       model.endswith("pro")
       or model.endswith("o3")
       or model.endswith("o4")
       or model.endswith("gpt-5")
    )

    return {
       "reasoning": reasoning,
       "notes": "Unknown model; capability inferred heuristically"
    }


def model_supports_thinking(model=None):
    return bool(get_model_info(model).get("reasoning"))

### 设置模型

In [ ]:
def set_model(model):
    global MODEL, THINKING_LEVEL
    
    MODEL = model
    
    if not model_supports_thinking(MODEL):
        THINKING_LEVEL = "off"
    
    if SESSION_FILE is not None:
        append_session_entry({
        "type": "model_change",
        "model": MODEL,
        })
    
    print(f"Model set to: {MODEL}")
    print(f"Thinking: {THINKING_LEVEL}")

### 设置 thinking

In [ ]:
def set_thinking(level):
    global THINKING_LEVEL
    
    if level not in THINKING_LEVELS:
        print(f"Invalid thinking level: {level}")
        print(f"Allowed: {THINKING_LEVELS}")
        return
    
    if level != "off" and not model_supports_thinking(MODEL):
        print(f"Model {MODEL} does not support thinking. Clamped to off.")
        THINKING_LEVEL = "off"
    else:
        THINKING_LEVEL = level
    
    if SESSION_FILE is not None:
        append_session_entry({
        "type": "thinking_change",
        "thinking": THINKING_LEVEL,
        })
    
    print(f"Thinking set to: {THINKING_LEVEL}")

### 替换call_model

In [538]:
def call_model():
    kwargs = {
       "model": MODEL,
       "messages": messages,
       "tools": tools,
       "tool_choice": "auto",
    }
    
    if THINKING_LEVEL != "off" and model_supports_thinking(MODEL):
        kwargs["reasoning_effort"] = THINKING_LEVEL
    
    try:
        response = client.chat.completions.create(**kwargs)
    except Exception as e:
       # 如果模型不接受 reasoning_effort，自动降级重试
        if "reasoning_effort" in str(e) and "reasoning_effort" in kwargs:
            print("[warning] model rejected reasoning_effort; retrying without it")
            kwargs.pop("reasoning_effort")
            response = client.chat.completions.create(**kwargs)
        else:
            raise
    
    msg = response.choices[0].message.model_dump(exclude_none=True)
    return msg

### 加命令 /model, /models, /thinking

In [539]:
if "_base_handle_command_without_model_thinking" not in globals():
    _base_handle_command_without_model_thinking = handle_command

def list_known_models():
    print("Known models:")
    for name, info in MODEL_REGISTRY.items():
        mark = "*" if name == MODEL else " "
        reasoning = "reasoning" if info.get("reasoning") else "no-reasoning"
        print(f"{mark} {name:16} {reasoning:14} {info.get('notes','')}")



def show_model_state():
    info = get_model_info(MODEL)
    print(f"MODEL: {MODEL}")
    print(f"supports thinking: {info.get('reasoning')}")
    print(f"THINKING_LEVEL: {THINKING_LEVEL}")


def handle_command(text):
    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_model_thinking(text)
        print("""
        Model commands:
        /models                  List known models
        /model                   Show current model
        /model NAME              Set model
        /thinking                Show thinking state
        /thinking LEVEL          Set thinking: off|minimal|low|medium|high
        """.strip())
        return result

    if cmd == "models":
        return list_known_models()

    if cmd == "model":
        if not arg:
            return show_model_state()
        return set_model(arg)

    if cmd == "thinking":
        if not arg:
            return show_model_state()
        return set_thinking(arg)

    return _base_handle_command_without_model_thinking(text)

### 测试

In [540]:
run("/models")

Known models:
* deepseek-v4-flash no-reasoning   Cheaper/faster model
  deepseek-v4-pro  reasoning      Reasoning-capable model


In [541]:
run("/model")

MODEL: deepseek-v4-flash
supports thinking: False
THINKING_LEVEL: off


In [542]:
run("/model deepseek-v4-pro")

Model set to: deepseek-v4-pro
Thinking: off


In [543]:
run("/thinking high")

Thinking set to: high


In [544]:
# run("用当前模型和当前thinking设置，说明你有哪些工具能力")

### 配置 Provider Registry

In [545]:
PROVIDER = globals().get("PROVIDER", "deepseek")
MODEL = globals().get("MODEL", "deepseek-chat")
THINKING_LEVEL = globals().get("THINKING_LEVEL", "off")

THINKING_LEVELS = ["off", "low", "medium", "high","xhigh"]

PROVIDER_REGISTRY = {
   "openai": {
       "api_key_env": "OPENAI_API_KEY",
       "base_url": None,
       "default_model": "gpt-4.1",
       "models": {
           "gpt-4.1": {"thinking": None},
           "gpt-4.1-mini": {"thinking": None},
           "gpt-5.5": {"thinking": "reasoning_effort"},
           "o3": {"thinking": "reasoning_effort"},
       },
   },

   "deepseek": {
       "api_key_env": "DEEPSEEK_API_KEY",
       "base_url": "https://api.deepseek.com",
       "default_model": "deepseek-v4-flash",
       "models": {

           # 如果你的 DeepSeek API 支持这些模型名，可以保留
           "deepseek-v4-flash": {"thinking": None},
           "deepseek-v4-pro": {"thinking": "reasoning_effort"},
       },
   },
}

### 根据 provider 创建 client

In [546]:
def get_provider_config(provider=None):
    provider = provider or PROVIDER
    if provider not in PROVIDER_REGISTRY:
        raise ValueError(f"Unknown provider: {provider}")
    return PROVIDER_REGISTRY[provider]


def get_llm_client(provider=None):
    cfg = get_provider_config(provider)
    api_key = os.environ.get(cfg["api_key_env"])
    
    if not api_key:
        raise RuntimeError(f"Missing API key env: {cfg['api_key_env']}")
    
    if cfg["base_url"]:
        return OpenAI(api_key=api_key, base_url=cfg["base_url"])
    
    return OpenAI(api_key=api_key)

In [547]:
os.environ.get("DEEPSEEK_API_KEY") is not None

True

### 获取当前模型能力

In [548]:
def get_current_model_info():
    cfg = get_provider_config()
    return cfg["models"].get(MODEL, {"thinking": None})


def model_supports_thinking():
    return get_current_model_info().get("thinking") is not None

In [549]:
get_current_model_info()

{'thinking': 'reasoning_effort'}

### 构建模型调用参数

In [550]:
def build_model_kwargs():
    kwargs = {
       "model": MODEL,
       "messages": messages,
       "tools": tools,
       "tool_choice": "auto",
    }
    
    thinking_mode = get_current_model_info().get("thinking")
    
    if THINKING_LEVEL == "off":
       return kwargs
    
    if thinking_mode == "reasoning_effort":
       kwargs["reasoning_effort"] = THINKING_LEVEL
    
    return kwargs

### 替换 call_model

In [551]:
def call_model():
    llm = get_llm_client()
    kwargs = build_model_kwargs()

    try:
        response = llm.chat.completions.create(**kwargs)
    except Exception as e:
        msg = str(e)

        # 某些 DeepSeek 模型不接受 reasoning_effort / thinking 参数
        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
        
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_body", None)
            
            response = llm.chat.completions.create(**kwargs)
        else:
            raise

    return response.choices[0].message.model_dump(exclude_none=True)

In [552]:
kwargs = build_model_kwargs()

In [553]:
llm = get_llm_client()
kwargs = build_model_kwargs()
response = llm.chat.completions.create(**kwargs)

In [554]:
response

ChatCompletion(id='9ba33251-5516-46c2-ae5d-4a844e66971b', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_lLv4CJY1FHcCW5ncxgb77933', function=Function(arguments='{"command": "cd D:\\\\workflow\\\\agent_harness && git log --oneline --all"}', name='bash'), type='function', index=0)], reasoning_content='The user wants me to look at the commit history (git log).'))], created=1785393194, model='deepseek-v4-pro', object='chat.completion', service_tier=None, system_fingerprint='fp_9954b31ca7_prod0820_fp8_kvcache_20260402', usage=CompletionUsage(completion_tokens=74, prompt_tokens=3597, total_tokens=3671, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=14, rejected_prediction_tokens=None), prompt_tokens_details=PromptToke

### Provider / Model 命令

In [555]:
def set_provider(provider):
    global PROVIDER, MODEL, THINKING_LEVEL

    if provider not in PROVIDER_REGISTRY:
        print(f"Unknown provider: {provider}")
        print(f"Available: {list(PROVIDER_REGISTRY.keys())}")
    return

    PROVIDER = provider
    MODEL = PROVIDER_REGISTRY[provider]["default_model"]
    THINKING_LEVEL = "off"

    if SESSION_FILE is not None:
        append_session_entry({
           "type": "provider_change",
           "provider": PROVIDER,
           "model": MODEL,
        })

    print(f"Provider set to: {PROVIDER}")
    print(f"Model set to: {MODEL}")


def set_model(model):
    global MODEL, THINKING_LEVEL

    MODEL = model

    if not model_supports_thinking():
        THINKING_LEVEL = "off"

    if SESSION_FILE is not None:
        append_session_entry({
           "type": "model_change",
           "provider": PROVIDER,
           "model": MODEL,
        })

    print(f"Model set to: {PROVIDER}/{MODEL}")
    print(f"Thinking: {THINKING_LEVEL}")


def set_thinking(level):
    global THINKING_LEVEL

    if level not in THINKING_LEVELS:
        print(f"Invalid thinking level: {level}")
        print(f"Allowed: {THINKING_LEVELS}")
        return

    if level != "off" and not model_supports_thinking():
        print(f"{PROVIDER}/{MODEL} does not support adjustable thinking. Clamped to off.")
        THINKING_LEVEL = "off"
    else:
        THINKING_LEVEL = level

    print(f"Thinking set to: {THINKING_LEVEL}")

### 追加命令系统

In [556]:

if "_base_handle_command_without_provider_adapter" not in globals():
    _base_handle_command_without_provider_adapter = handle_command


def handle_command(text):
    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_provider_adapter(text)
        print("""
        Provider commands:
        /providers              List providers
        /provider               Show current provider
        /provider NAME          Set provider
        /models                 List models for current provider
        /model                  Show current model
        /model NAME             Set model
        /thinking               Show thinking level
        /thinking LEVEL         Set thinking: off|minimal|low|medium|high
        """.strip())
        return result

    if cmd == "providers":
        for name in PROVIDER_REGISTRY:
            mark = "*" if name == PROVIDER else " "
            print(f"{mark} {name}")
        return

    if cmd == "provider":
        if not arg:
            print(f"Provider: {PROVIDER}")
            print(f"Model: {MODEL}")
            return
        return set_provider(arg)

    if cmd == "models":
        cfg = get_provider_config()
        for name, info in cfg["models"].items():
            mark = "*" if name == MODEL else " "
            thinking = info.get("thinking") or "none"
            print(f"{mark} {name:24} thinking={thinking}")
        return

    if cmd == "model":
        if not arg:
            print(f"Current model: {PROVIDER}/{MODEL}")
            print(f"Thinking: {THINKING_LEVEL}")
            return
        return set_model(arg)

    if cmd == "thinking":
        if not arg:
            print(f"Thinking: {THINKING_LEVEL}")
            print(f"Model supports thinking:{model_supports_thinking()}")
            return
        return set_thinking(arg)

    return _base_handle_command_without_provider_adapter(text)

### 测试

In [557]:
run("/provider deepseek")

In [558]:
run("/models")

  deepseek-v4-flash        thinking=none
* deepseek-v4-pro          thinking=reasoning_effort


In [559]:
run("/model")

Current model: deepseek/deepseek-v4-pro
Thinking: high


In [560]:
run("简单分析，为啥那么 agent harness 需要工具调用循环")

**Tool call:** `read_file`

```text
1: 本项目学习Pi agent的harness框架，从简单到复杂，构建一个类似的python版本，通过实践来学习agent harness
```

**Tool call:** `read_file`

```text
1: # Hello Mini Harness 🏗️
2: 
3: ## 简介
4: 
5: **`mini_harness.ipynb`** 是当前项目中最核心的实验 Notebook，实现了一个**轻量级 Agent Harness 框架**的 Python 原型，用于学习 LLM Agent 的工具调用循环。
6: 
7: ## 核心组件
8: 
9: | 组件 | 作用 |
10: |------|------|
11: | **`messages[]`** | 维护 system/user/assistant/tool 多轮对话历史 |
12: | **`call_model()`** | 调用 DeepSeek LLM API，支持 tool_choice="auto" |
13: | **`execute_tool_call()`** | 解析模型返回的工具调用并执行 |
14: | **`ask()`** | 主循环入口，自动多轮交互直到模型直接回复或超限 |
15: | **`tool_impls{}`** | 工具名 → 函数实现的注册表 |
16: 
17: ## 已实现的工具
18: 
19: - **`list_files(path)`** 🔓 — 列出工作区目录
20: - **`read_file(path, offset, limit)`** 🔓 — 分页读取文件
21: - **`write_file(path, content)`** 🔒 — 写文件（需用户确认）
22: 
23: ## 安全机制
24: 
25: - **路径逃逸检测** — `resolve_workspace_path()` 使用 `relative_to(ROOT)` 校验
26: - **写操作确认** — 写文件前通过控制台 `input()` 征求用户授权
27: 
28: ## 构建路线
29: 
30: ```
31: Phase 1 ─── 最小原型: messages + list_files + call_model
32:      │
33:      ▼
34: Phase 2 ─── Agent 循环: ask() 实现多轮 tool-use loop
35:      │
36:      ▼
37: Phase 3 ─── 增强工具: 添加 read_file + 路径安全守卫
38:      │
39:      ▼
40: Phase 4 ─── 写能力: write_file + 用户确认机制
41: ```
42: 
43: > 🚀 通过从简单到复杂逐步构建，以实践理解 Agent Harness 的核心模式！
```

**Tool call:** `read_file`

...

**Tool call:** `bash`

```text
$ cd D:\workflow\agent_harness && python -c "
import json, sys
nb=json.load(open('mini_harness.ipynb','r',encoding='utf-8'))
for idx in range(10):
    c=nb['cells'][idx]
    src=''.join(c['source'])
    line = f'{idx} | {c[\"cell_type\"]} | {len(src)} | {\"def\" in src}'
    sys.stdout.write(line+'\\n')
sys.stdout.flush()
"
       exit_code: 0

       STDOUT:
       

       STDERR:
       
       
```

**Tool call:** `write_file`


Tool `write_file` requests permission:
Reason: file modification
Permission key: write_file:_inspect_nb.py
Args:
{
  "path": "_inspect_nb.py",
  "content": "import json, sys\n\nnb = json.load(open('mini_harness.ipynb', 'r', encoding='utf-8'))\n\nfor idx, cell in enumerate(nb['cells']):\n    src = ''.join(cell['source'])\n    if cell['cell_type'] != 'code' or len(src) == 0:\n        continue\n    if 'def ' in src or 'for ' in src or 'while ' in src:\n        print(f'=== Cell {idx} ({cell[\"cell_type\"]}) ===')\n        print(src[:500])\n        print()\n"
}
Choose:
    y  = allow once
    s  = allow for this session
    n  = deny


KeyboardInterrupt: Interrupted by user

## streaming 输出

### 合并流式 tool_call 片段

In [ ]:
STREAMING_ENABLED = True

def merge_tool_call_delta(tool_calls_by_index, delta_tool_call):
    d = delta_tool_call.model_dump(exclude_none=True) if hasattr(delta_tool_call, "model_dump") else delta_tool_call

    idx = d.get("index", len(tool_calls_by_index))

    acc = tool_calls_by_index.setdefault(idx, {
    "id": None,
    "type": "function",
    "function": {
    "name": None,
    "arguments": "",
    },
    })

    if d.get("id"):
        acc["id"] = d["id"]

    if d.get("type"):
        acc["type"] = d["type"]

    fn = d.get("function") or {}

    if fn.get("name"):
        acc["function"]["name"] = fn["name"]

    if fn.get("arguments"):
        acc["function"]["arguments"] += fn["arguments"]


### 完成 tool_calls

In [ ]:
def finalize_streamed_tool_calls(tool_calls_by_index):
    result = []

    for idx in sorted(tool_calls_by_index):
        tc = tool_calls_by_index[idx]

        if not tc.get("id"):
            tc["id"] = f"call_{uuid.uuid4().hex[:8]}"
    
        if not tc["function"].get("name"):
            tc["function"]["name"] = ""
    
        if tc["function"].get("arguments") is None:
            tc["function"]["arguments"] = "{}"
    
        result.append(tc)

    return result

### 实现流式模型调用

In [ ]:
def call_model_streaming():
    llm = get_llm_client()
    kwargs = build_model_kwargs()
    kwargs["stream"] = True

    try:
        stream = llm.chat.completions.create(**kwargs)
    
    except Exception as e:
        msg = str(e)

        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
            
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_body", None)
            
            stream = llm.chat.completions.create(**kwargs)
        else:
            raise

    
    role = "assistant"
    content_parts = []
    tool_calls_by_index = {}
    
    for chunk in stream:
        if not chunk.choices:
            continue
    
        choice = chunk.choices[0]
        delta = choice.delta
    
        if delta is None:
            continue
    
        d = delta.model_dump(exclude_none=True)
    
        if d.get("role"):
            role = d["role"]
    
        if d.get("content"):
            text = d["content"]
            content_parts.append(text)
            print(text, end="", flush=True)
    
        for tc in d.get("tool_calls") or []:
            merge_tool_call_delta(tool_calls_by_index, tc)
        
    content = "".join(content_parts)

    if content:
        print()

    assistant_msg = {
        "role": role,
        "content": content if content else None,
    }
    
    tool_calls = finalize_streamed_tool_calls(tool_calls_by_index)

    if tool_calls:
        assistant_msg["tool_calls"] = tool_calls
    
    return assistant_msg
        

### 统一模型调用入口

In [ ]:
def call_model_agent():
    if not STREAMING_ENABLED:
        return call_model(), False
    
    try:
        return call_model_streaming(), True
    
    except Exception as e:
        print(f"\n[streaming warning] {type(e).__name__}: {e}")
        print("[streaming warning] fallback to non-streaming call")
        return call_model(), False

### 替换 ask

In [575]:
def pending_tool_calls(msgs):
    pending = []

    for index, msg in enumerate(msgs):
        role = msg.get("role")

        if pending and role != "tool":
            raise ValueError(
                f"Invalid message history at message {index}: "
                f"{role} appeared before tool responses: {pending}"
            )

        if role == "assistant":
            pending = [
                tc.get("id")
                for tc in msg.get("tool_calls") or []
            ]

            if any(not tool_call_id for tool_call_id in pending):
                raise ValueError(
                    f"Assistant message {index} contains a tool call without id"
                )

        elif role == "tool":
            tool_call_id = msg.get("tool_call_id")

            if tool_call_id not in pending:
                raise ValueError(
                    f"Orphan tool response at message {index}: "
                    f"{tool_call_id}"
                )

            pending.remove(tool_call_id)

    return pending


def assert_complete_tool_history(msgs):
    pending = pending_tool_calls(msgs)

    if pending:
        raise ValueError(
            "Cannot continue from an incomplete tool-call turn: "
            + ", ".join(pending)
        )

def ask(prompt, max_turns=5000):
    if SESSION_FILE is None:
        raise RuntimeError("No active session. Run new_session() or resume_session(path) first.")

    user_msg = {
    "role": "user",
    "content": prompt,
    }

    messages.append(user_msg)
    save_message(user_msg)

    for turn in range(max_turns):
        maybe_auto_compact(reason="before_model_call")
        assert_complete_tool_history(messages)

        assistant_msg, already_rendered = call_model_agent()
        
        messages.append(assistant_msg)
        save_message(assistant_msg)
        
        content = assistant_msg.get("content")
        if content and not already_rendered:
            display(Markdown(content))

        tool_calls = assistant_msg.get("tool_calls") or []

        if not tool_calls:
            return assistant_msg

        for tool_call in tool_calls:
            name = tool_call["function"]["name"]
            
            display(Markdown(f"**Tool call:** `{name}`"))
            
            result = execute_tool_call(tool_call)
            
            display(Markdown(f"```text\n{result[:3000]}\n```"))
            
            tool_msg = {
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "content": result,
            }

            messages.append(tool_msg)
            save_message(tool_msg)

    raise RuntimeError("Agent exceeded max_turns")

### 加 /streaming 命令

In [576]:
if "_base_handle_command_without_streaming" not in globals():
    _base_handle_command_without_streaming = handle_command


def handle_command(text):
    global STREAMING_ENABLED

    parts = text.strip().split(maxsplit=1)
    cmd = parts[0][1:]
    arg = parts[1].strip() if len(parts) > 1 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_streaming(text)
        print("""
        Streaming commands:
        /streaming          Show streaming status
        /streaming on       Enable streaming
        /streaming off      Disable streaming
        """.strip())
        return result

    if cmd == "streaming":
        if not arg:
            print(f"STREAMING_ENABLED: {STREAMING_ENABLED}")
            return
    
        if arg == "on":
            STREAMING_ENABLED = True
            print("Streaming enabled.")
            return
    
        if arg == "off":
            STREAMING_ENABLED = False
            print("Streaming disabled.")
            return
    
        print("Usage: /streaming [on|off]")
        return

    return _base_handle_command_without_streaming(text)

### 测试

In [ ]:
run("/streaming on")

In [ ]:
run("用三句话说明这个mini harness有哪些能力")

### Retry 配置

In [ ]:
import random
import time

In [ ]:
# configuration
RETRY_ENABLED = True
RETRY_MAX_ATTEMPTS = 3
RETRY_BASE_DELAY = 2.0
RETRY_MAX_DELAY = 30.0

### Error classification

In [ ]:
def get_error_status_code(error):
    for attr in ("status_code", "status"):
        value = getattr(error, attr, None)
        if isinstance(value, int):
            return value

    response = getattr(error, "response", None)
    value = getattr(response, "status_code", None)
    if isinstance(value,int):
        return value
    return None

def classify_model_error(error):
    status = get_error_status_code(error)
    text = f"{type(error).__name__}: {error}".lower()

    context_patterns = [
    "context length",
    "maximum context",
    "too many tokens",
    "token limit",
    "input is too long",
    ]

    if any(p in text for p in context_patterns):
        return {
            "retryable": False,
            "reason": "context_overflow",
        }
    non_retry_patterns = [
   "invalid api key",
   "unauthorized",
   "authentication",
   "permission denied",
   "forbidden",
   "model not found",
   "insufficient_quota",
   "invalid_request",
    ]
    if status in (400,401,403,404):
        return {
            "retryable": False,
            "reason": f"non_retryable_status_{status}",
        }

    if any(p in text for p in non_retry_patterns):
        return {
            "retryable": False,
            "reason": "non_retryable_error",
        }

    if status in (408, 409, 425, 429, 500, 502, 503, 504, 529):
        return {
            "retryable": True,
            "reason": f"retryable_status_{status}",
        }
    retry_patterns = [
        "fetch failed",
        "connection error",
        "connection reset",
        "econnreset",
        "etimedout",
        "timed out",
        "timeout",
        "enotfound",
        "eai_again",
        "temporarily unavailable",
        "overloaded",
        "rate limit",
        "429",
        "500",
        "502",
        "503",
        "504",
        "529",
    ]

    if any(p in text for p in retry_patterns):
        return {
            "retryable": True,
            "reason": "transient_network_or_provider_error",
        }

    return {
        "retryable": False,
        "reason": "unknown_non_retryable",
    }    

### Retry delay

In [568]:
def get_retry_after_seconds(error):
    headers = None
    
    response = getattr(error, "response", None)
    if response is not None:
        headers = getattr(response, "headers", None)
    
    if headers is None:
        headers = getattr(error, "headers", None)
    
    if not headers:
        return None
    
    retry_after = None
    try:
        retry_after = headers.get("retry-after") or headers.get("Retry-After")
    except Exception:
        return None

    if retry_after is None:
        return None

    
    try:
        return float(retry_after)
    except Exception:
        return None

def compute_retry_delay(error, attempt):
    retry_after = get_retry_after_seconds(error)

    if retry_after is not None:
        return min(float(retry_after), RETRY_MAX_DELAY)
    base = RETRY_BASE_DELAY*(2**(attempt -1 ))
    base = min(base, RETRY_MAX_DELAY)

    jitter = 0.8 + random.random()*0.4
    return base*jitter
    

### 保存 retry 事件到 session

In [ ]:
def save_retry_event(event):
    try:
        if globals().get("SESSION_FILE") is not None:
            append_session_entry({
                "type": "retry",
                **event,
            })
    except Exception:
        pass

### 通用 retry wrapper

In [ ]:
def call_with_retry(fn, label="model_call"):
    if not RETRY_ENABLED:
        return fn()

    last_error = None
    
    for attempt in range(1, RETRY_MAX_ATTEMPTS + 1):
        try:
            if attempt > 1:
                print(f"[retry] {label}: attempt {attempt}/{RETRY_MAX_ATTEMPTS}")
            result = fn()

            if attempt > 1:
                save_retry_event({
                    "event": "retry_success",
                    "label": label,
                    "attemp": attempt,
                })
            return result

        except Exception as e:
            last_error = e
            info = classify_model_error(e)

            save_retry_event({
                "event": "retry_error",
                "label": label,
                "attempt": attempt,
                "retryable": info["retryable"]
            })

            if not info["retryable"]:
                print(f"[retry] not retryable: {info['reason']}")
                raise
    
            if attempt >= RETRY_MAX_ATTEMPTS:
                print(f"[retry] exhausted after {attempt} attempts")
                raise
    
            delay = compute_retry_delay(e, attempt)
    
            print(f"[retry] {label} failed: {info['reason']}")
            print(f"[retry] error: {e}")
            print(f"[retry] sleeping {delay:.1f}s")
    
            time.sleep(delay)

    raise last_error
                
    

### 包装已有模型

In [ ]:
if "_call_model_without_retry" not in globals():
    _call_model_without_retry = call_model

    def call_model():
        provider = globals().get("PROVIDER", "unknown")
        model = globals().get("MODEL", "unknown")
        return call_with_retry(
            _call_model_without_retry,
            label=f"{provider}/{model}:non_streaming",
        )

if "call_model_streaming" in globals():
    if "_call_model_streaming_without_retry" not in globals():
        _call_model_streaming_without_retry = call_model_streaming

    def call_model_streaming():
        provider = globals().get("PROVIDER", "unknown")
        model = globals().get("MODEL", "unknown")
        return call_with_retry(
            _call_model_streaming_without_retry,
            label=f"{provider}/{model}:streaming",
        )

### 增加 /retry 命令

In [ ]:
if "_base_handle_command_without_retry" not in globals():
    _base_handle_command_without_retry = handle_command


def retry_status():
    print(f"RETRY_ENABLED: {RETRY_ENABLED}")
    print(f"RETRY_MAX_ATTEMPTS: {RETRY_MAX_ATTEMPTS}")
    print(f"RETRY_BASE_DELAY: {RETRY_BASE_DELAY}")
    print(f"RETRY_MAX_DELAY: {RETRY_MAX_DELAY}")

    

def handle_command(text):
    global RETRY_ENABLED
    global RETRY_MAX_ATTEMPTS
    global RETRY_BASE_DELAY
    global RETRY_MAX_DELAY
    
    parts = text.strip().split(maxsplit=2)
    cmd = parts[0][1:]
    arg1 = parts[1].strip() if len(parts) > 1 else ""
    arg2 = parts[2].strip() if len(parts) > 2 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_retry(text)
        print("""
        Retry commands:
        /retry                    Show retry status
        /retry on                 Enable retry
        /retry off                Disable retry
        /retry attempts N         Set max attempts
        /retry delay N            Set base delay seconds
        /retry max-delay N        Set max delay seconds
        """.strip())
        return result

    if cmd == "retry":
        if not arg1:
            return retry_status()

        if arg1 == "on":
            RETRY_ENABLED = True
            print("Retry enabled.")
            return 

        if arg1 == "off":
            RETRY_ENABLED = False
            print("Retry disable.")
            return 

        if arg1 == "attempts" and arg2:
            RETRY_MAX_ATTEMPTS = int(arg2)
            print(f"RETRY_MAX_ATTEMPTS = {RETRY_MAX_ATTEMPTS}")
            return

        if arg1 == "delay" and arg2:
            RETRY_BASE_DELAY = float(arg2)
            print(f"RETRY_BASE_DELAY = {RETRY_BASE_DELAY}")
            return

        if arg1 == "max-delay" and arg2:
            RETRY_MAX_DELAY = float(arg2)
            print(f"RETRY_MAX_DELAY = {RETRY_MAX_DELAY}")
            return

        print("Usage: /retry [on|off|attempts N|delay N|max-delay N]")

        return 

    return _base_handle_command_without_retry(text)
    
    

### 测试

In [ ]:
class FakeFetchFailed(Exception):
    pass

def test_retry_wrapper():
    counter = {"n": 0}

    def flaky():
        counter["n"] += 1
        if counter["n"] < 3:
            raise FakeFetchFailed("fetch failed")
        return "ok"
    return call_with_retry(flaky, label = "fake_fetch_failed")

In [ ]:
test_retry_wrapper()

In [ ]:
# run("/provider deepseek")
# run("/model deepseek-v4-flash")
# run("/streaming off")
# run("用一句确认当前agent 可以正常调用deepseek")

## 更加完整文件工具 ls/find/grep

### 增强输出截断

In [ ]:
TOOL_OUTPUT_DIR = ROOT / ".mini-pi" / "tool-outputs"
TOOL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def truncate_text(text, max_chars=12000):
    text = str(text)
    
    if len(text) <= max_chars:
        return text
    
    path = TOOL_OUTPUT_DIR / f"output-{uuid.uuid4().hex[:8]}.txt"
    path.write_text(text, encoding="utf-8")
    
    return (
        text[:max_chars]
        + f"\n\n[truncated: {len(text)} chars total. Full output saved to {path}]"
        )

### 文件遍历 helper

In [ ]:
import os
import fnmatch
import re

DEFAULT_EXCLUDE_DIRS = {
".git",
".mini-pi",
"__pycache__",
".pytest_cache",
".mypy_cache",
".ruff_cache",
"node_modules",
".venv",
"venv",
"dist",
"build",
}
def is_hidden_path(p):
    try:
        rel = p.resolve().relative_to(ROOT.resolve())
    except ValueError:
        return True
    
    return any(part.startswith(".") for part in rel.parts)

def should_skip_path(p, include_hidden=False):
    parts = set(p.parts)
    
    if parts & DEFAULT_EXCLUDE_DIRS:
        return True
    
    if not include_hidden and is_hidden_path(p):
        return True
    
    return False

def relpath(p):
    return str(p.resolve().relative_to(ROOT.resolve()))

def is_probably_binary(path, sample_size=2048):
    try:
        data = Path(path).read_bytes()[:sample_size]
        return b"\0" in data
    except Exception:
        return True      


### 实现ls

In [ ]:
def tool_ls(path=".", depth=1, max_entries=200,include_hidden=False):
    p = resolve_workspace_path(path)
    
    depth = max(1, min(int(depth), 5))
    max_entries = max(1, min(int(max_entries), 1000))
    
    if not p.exists():
        return f"ERROR: path does not exist: {path}"

    if p.is_file():
        return relpath(p)

    lines = []
    count = 0

    def walk(directory, prefix="", current_depth=1):
        nonlocal count

        entries = sorted(
        directory.iterdir(),
        key=lambda x: (not x.is_dir(), x.name.lower())
        )

        for child in entries:
            if should_skip_path(child, include_hidden=include_hidden):
                continue

            suffix = "/" if child.is_dir() else ""
            lines.append(f"{prefix}{child.name}{suffix}")
            count += 1

            if count >= max_entries:
                lines.append(f"[truncated: reachedmax_entries={max_entries}]")
                return True

            if child.is_dir() and current_depth < depth:
                if walk(child, prefix + "  ", current_depth + 1):
                    return True

        return False

    walk(p)

    if not lines:
        return "<empty>"

    return truncate_text("\n".join(lines))

### 实现 find

In [ ]:
def tool_find(path=".", pattern="*", type="any", max_results=200, include_hidden=False):
    base = resolve_workspace_path(path)

    if not base.exists():
        return f"ERROR: path does not exist: {path}"

    max_results = max(1, min(int(max_results), 1000))

    if type not in ("any", "file", "dir"):
        return "ERROR: type must be one of: any, file, dir"

    results = []

    for root, dirs, files in os.walk(base):
        root_path = Path(root)

        dirs[:] = [
        d for d in dirs
        if not should_skip_path(root_path / d,
        include_hidden=include_hidden)
        ]

        candidates = []

        if type in ("any", "dir"):
            candidates.extend(root_path / d for d in dirs)

        if type in ("any", "file"):
            candidates.extend(root_path / f for f in files)

        for candidate in candidates:
            if should_skip_path(candidate,include_hidden=include_hidden):
                continue

            r = relpath(candidate)

            if fnmatch.fnmatch(candidate.name, pattern) or fnmatch.fnmatch(r, pattern):
                results.append(r + ("/" if candidate.is_dir() else ""))

                if len(results) >= max_results:
                    results.append(f"[truncated: reached max_results={max_results}]")
                    return "\n".join(results)

    return "\n".join(results) if results else "<no matches>"

### 实现 grep

In [ ]:
def tool_grep(
pattern,
path=".",
file_glob="*",
case_sensitive=True,
max_results=200,
include_hidden=False,
):
    base = resolve_workspace_path(path)
    
    if not base.exists():
        return f"ERROR: path does not exist: {path}"
    
    flags = 0 if case_sensitive else re.IGNORECASE
    
    try:
        regex = re.compile(pattern, flags)
    except Exception as e:
        return f"ERROR: invalid regex: {e}"
    
    max_results = max(1, min(int(max_results), 1000))
    results = []
    
    def candidate_files():
        if base.is_file():
            yield base
            return
    
        for root, dirs, files in os.walk(base):
            root_path = Path(root)
    
            dirs[:] = [
            d for d in dirs
            if not should_skip_path(root_path / d,
            include_hidden=include_hidden)
            ]
    
            for name in files:
                p = root_path / name
            
                if should_skip_path(p,include_hidden=include_hidden):
                    continue
    
                r = relpath(p)
    
                if fnmatch.fnmatch(name, file_glob) or fnmatch.fnmatch(r, file_glob):
                    yield p
    
    for file_path in candidate_files():
        if is_probably_binary(file_path):
            continue
    
        try:
            text = file_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        
        for line_no, line in enumerate(text.splitlines(), start=1):
            if regex.search(line):
                short_line = line[:500]
                results.append(f"{relpath(file_path)}:{line_no}: {short_line}")
        
                if len(results) >= max_results:
                    results.append(f"[truncated: reached max_results={max_results}]")
                    return truncate_text("\n".join(results))
        
    return truncate_text("\n".join(results)) if results else "<no matches>"

### 注册工具

In [ ]:
tool_impls["ls"] = tool_ls
tool_impls["find"] = tool_find
tool_impls["grep"] = tool_grep

# 避免重复 schema 的 helper
def upsert_tool_schema(schema):
    name = schema["function"]["name"]

    for i, t in enumerate(tools):
        if t.get("function", {}).get("name") == name:
            tools[i] = schema
            return
    
    tools.append(schema)


### 注册 ls/ find / grep schema

In [ ]:
upsert_tool_schema({
"type": "function",
"function": {
"name": "ls",
"description": "List directory contents as a compact tree.",
"parameters": {
"type": "object",
"properties": {
"path": {"type": "string", "default": "."},
"depth": {"type": "integer", "default": 1},
"max_entries": {"type": "integer", "default": 200},
"include_hidden": {"type": "boolean", "default":
False},
},
"required": [],
"additionalProperties": False,
},
},
})


upsert_tool_schema({
"type": "function",
"function": {
"name": "find",
"description": "Find files or directories by name/glob pattern.",
"parameters": {
"type": "object",
"properties": {
"path": {"type": "string", "default": "."},
"pattern": {"type": "string", "default": "*"},
"type": {
   "type": "string",
   "enum": ["any", "file", "dir"],
   "default": "any",
},
"max_results": {"type": "integer", "default": 200},
"include_hidden": {"type": "boolean", "default":
False},
},
"required": [],
"additionalProperties": False,
},
},
})

upsert_tool_schema({
"type": "function",
"function": {
"name": "grep",
"description": "Search file contents using a regex pattern.",
"parameters": {
"type": "object",
"properties": {
"pattern": {"type": "string"},
"path": {"type": "string", "default": "."},
"file_glob": {"type": "string", "default": "*"},
"case_sensitive": {"type": "boolean", "default":
True},
"max_results": {"type": "integer", "default": 200},
"include_hidden": {"type": "boolean", "default":
False},
},
"required": ["pattern"],
"additionalProperties": False,
},
},
})

### 更新权限策略：这些是只读工具

In [ ]:
READ_ONLY_TOOLS = {
"list_files",
"read_file",
"ls",
"find",
"grep",
}

In [ ]:

def permission_key_for_tool(tool, args):
    if tool in READ_ONLY_TOOLS:
        return f"{tool}:readonly"

    if tool in ("write_file", "edit_file"):
        return f"{tool}:{args.get('path', '')}"

    if tool == "bash":
        command = args.get("command", "").strip()
        first_word = command.split()[0] if command.split() else ""
        return f"bash:{first_word}"
    
    return f"{tool}:default"


def assess_tool_permission(tool, args):
    key = permission_key_for_tool(tool, args)

    if key in SESSION_ALLOWED_TOOL_PATTERNS:
        return {
        "action": PERMISSION_ALLOW,
        "reason": f"allowed for session: {key}",
        "key": key,
        }

    if tool in READ_ONLY_TOOLS:
        return {
        "action": PERMISSION_ALLOW,
        "reason": "read-only tool",
        "key": key,
        }

    if tool in ("write_file", "edit_file"):
        return {
        "action": PERMISSION_CONFIRM,
        "reason": "file modification",
        "key": key,
        }

    if tool == "bash":
        command = args.get("command", "")

        blocked = [
        "rm -rf",
        "sudo",
        "mkfs",
        "dd if=",
        "chmod -R 777",
        "chown -R",
        ":(){",
        ]

        for pattern in blocked:
            if pattern in command:
                return {
                "action": PERMISSION_BLOCK,
                "reason": f"blocked dangerous bash pattern: {pattern}",
                "key": key,
                }

        return {
        "action": PERMISSION_CONFIRM,
        "reason": "bash command",
        "key": key,
        }

    return {
    "action": PERMISSION_CONFIRM,
    "reason": "unknown tool risk",
    "key": key,
    }

### 更新系统提示词

In [ ]:
messages[0]["content"] = f"""
You are a minimal coding agent running inside Jupyter Notebook.

Workspace root:
{ROOT}

Available tools:
- ls: list directory contents as a compact tree.
- find: find files or directories by glob pattern.
- grep: search file contents by regex.
- read_file: read a text file.
- write_file: create or overwrite a text file.
- edit_file: edit a file by exact text replacement.
- bash: run shell commands in the workspace.

Rules:
- Prefer ls/find/grep/read_file for inspection before using bash.
- Use read_file before modifying existing files.
- Use edit_file for targeted modifications.
- oldText in edit_file must match exactly once.
- Use write_file only when creating a new file or replacing a full
file.
- Avoid destructive commands.
- Be concise.
""".strip()

### 测试

In [ ]:
tool_ls(".", depth=2)

In [ ]:
tool_find(".",pattern="*.py",type="file")

In [ ]:
tool_grep("TODO|FIXME", path=".", file_glob="*.py", case_sensitive=False)

In [ ]:
run("请使用 ls/find/grep 检查当前 workspace 的主要文件，并总结项目结构。")

## Resource loader: 自动加载项目资源

In [ ]:
# 目标：
# AGENTS.md                 项目指令
# .mini-pi/settings.json    配置
# .mini-pi/prompts/*.md     prompt templates
# .mini-pi/skills/*/SKILL.md skills

### 目录和全局状态

In [ ]:
MINI_PI_DIR = ROOT / ".mini-pi"
PROMPTS_DIR = MINI_PI_DIR / "prompts"
SKILLS_DIR = MINI_PI_DIR / "skills"
SETTINGS_FILE = MINI_PI_DIR / "settings.json"

PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
SKILLS_DIR.mkdir(parents=True, exist_ok=True)

RESOURCES = {
   "settings": {},
   "agents_files": [],
   "prompts": {},
   "skills": {},
}


### settings loader

In [ ]:
def default_settings():
    return {    
        "provider": globals().get("PROVIDER", "deepseek"),
        "model": globals().get("MODEL", "deepseek-chat"),
        "thinking": globals().get("THINKING_LEVEL", "off"),
        "streaming": globals().get("STREAMING_ENABLED", True),
        "autoCompact": {  
            "enabled": globals().get("AUTO_COMPACT_ENABLED", True),
            "charLimit": globals().get("AUTO_COMPACT_CHAR_LIMIT", 80000),
            "keepRecent": globals().get("AUTO_COMPACT_KEEP_RECENT",12),
            },
        "retry": {
            "enabled": globals().get("RETRY_ENABLED", True),
            "maxAttempts": globals().get("RETRY_MAX_ATTEMPTS", 3),
            "baseDelay": globals().get("RETRY_BASE_DELAY", 2.0),
            "maxDelay": globals().get("RETRY_MAX_DELAY", 30.0),
        },
    }

def load_settings():
    if not SETTINGS_FILE.exists():
        settings = default_settings()
        SETTINGS_FILE.write_text(
        json.dumps(settings,ensure_ascii=False,indent=2),
        encoding="utf-8",
        )
        return settings
    return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))

def apply_settings(settings):
    global PROVIDER, MODEL, THINKING_LEVEL
    global STREAMING_ENABLED
    global AUTO_COMPACT_ENABLED, AUTO_COMPACT_CHAR_LIMIT, AUTO_COMPACT_KEEP_RECENT
    global RETRY_ENABLED, RETRY_MAX_ATTEMPTS, RETRY_BASE_DELAY, RETRY_MAX_DELAY

    if "provider" in settings:
        PROVIDER = settings["provider"]
    
    if "model" in settings:
        MODEL = settings["model"]
    
    if "thinking" in settings:
        THINKING_LEVEL = settings["thinking"]
    
    if "streaming" in settings:
        STREAMING_ENABLED = bool(settings["streaming"])

    ac = settings.get("autoCompact", {})
    AUTO_COMPACT_ENABLED = bool(ac.get("enabled", globals().get("AUTO_COMPACT_ENABLED", True)))
    AUTO_COMPACT_CHAR_LIMIT = int(ac.get("charLimit", globals().get("AUTO_COMPACT_CHAR_LIMIT", 80000)))
    AUTO_COMPACT_KEEP_RECENT = int(ac.get("keepRecent", globals().get("AUTO_COMPACT_KEEP_RECENT", 12)))
    
    rt = settings.get("retry", {})
    RETRY_ENABLED = bool(rt.get("enabled", globals().get("RETRY_ENABLED", True)))
    RETRY_MAX_ATTEMPTS = int(rt.get("maxAttempts", globals().get("RETRY_MAX_ATTEMPTS", 3)))
    RETRY_BASE_DELAY = float(rt.get("baseDelay", globals().get("RETRY_BASE_DELAY", 2.0)))
    RETRY_MAX_DELAY = float(rt.get("maxDelay", globals().get("RETRY_MAX_DELAY", 30.0)))

### 加载 AGENT.md

In [ ]:
def load_agents_files():
    files = []

    for path in [
        ROOT / "AGENT.md",
        ROOT / "CLAUDE.md",
        MINI_PI_DIR/ "AGENTS.md",
    ]:
        if path.exists() and path.is_file():
            files.append({
                "path": str(path),
                "content": path.read_text(encoding="utf-8", errors="replace"),
            })
    return files     

### 加载 prompt templates

In [ ]:
def load_prompt_templates():
    prompts = {}
    
    for path in sorted(PROMPTS_DIR.glob("*.md")):
        name = path.stem
        content = path.read_text(encoding="utf-8", errors="replace")
        
        prompts[name] = {
           "name": name,
           "path": str(path),
           "content": content,
        }
    
    return prompts

def expand_prompt_template(name,args = ""):
    prompt = RESOURCES["prompts"].get(name)
    if not prompt:
        raise ValueError(f"Unknown prompt template: {name}")

    content = prompt["content"]
    content = content.replace("{{input}}", args)
    content = content.replace("{{args}}", args)

    if "{{input}}" not in prompt["content"] and "{{args}}" not in prompt["content"] and args:
        content = content.rstrip() + "\n\n" + args
    return content

### 加载 skills

In [569]:
def load_skills():
    skills = {}
    for skill_file in sorted(SKILLS_DIR.glob("*/SKILL.md")):
        name = skill_file.parent.name
        content = skill_file.read_text(encoding="utf-8",errors = "replace")
        skills[name] = {
        "name": name,
        "path": str(skill_file),
        "content": content,
        }
    return skills

def expand_skill(name, user_message=""):
    skill = RESOURCES["skills"].get(name)

    if not skill:
        raise ValueError(f"Unknown skill: {name}")

    text = (
        f'<skill name="{name}" location="{skill["path"]}">\n'
        f'{skill["content"]}\n'
        "</skill>"
    )

    if user_message:
        text += "\n\n" + user_message

    return text
    

### 重建 system prompt

In [ ]:
def rebuild_system_prompt():
    tool_names = sorted(tool_impls.keys()) if "tool_impls" in globals() else []
    parts = [
    f"""
    You are a minial coding agent running inside Jupyter Notebook.

    Workspace root:
    {ROOT}

    Available tools:
    {chr(10).join("- " + name for name in tool_names)}

    Rules:
   - Prefer structured tools like ls/find/grep/read_file before bash.
   - Use read_file before modifying existing files.
   - Use edit_file for targeted modifications.
   - oldText in edit_file must match exactly once.
   - Use write_file only for new files or full rewrites.
   - Avoid destructive commands.
   - Be concise.
   """.strip()
    ]

    if RESOURCES["agents_files"]:
        parts.append("Project instructions: ")
        for f in RESOURCES["agents_files"]:
            parts.append(f"## {f['path']}\n\n{f['content']}")

    return "\n\n---\n\n".join(parts)

def apply_system_prompt():
    system_prompt = rebuild_system_prompt()

    if messages and messages[0].get("role") == "system":
        messages[0]["content"] = system_prompt
    else:
        messages.insert(0,{"role": "system", "content": system_prompt})

    

### reload resources

In [ ]:
def reload_resources():
    settings = load_settings()
    apply_settings(settings)
    
    RESOURCES["settings"] = settings
    RESOURCES["agents_files"] = load_agents_files()
    RESOURCES["prompts"] = load_prompt_templates()
    RESOURCES["skills"] = load_skills()
    
    apply_system_prompt()
    
    print("Resources reloaded.")
    print(f"settings: {SETTINGS_FILE}")
    print(f"agents files: {len(RESOURCES['agents_files'])}")
    print(f"prompts: {list(RESOURCES['prompts'].keys())}")
    print(f"skills: {list(RESOURCES['skills'].keys())}")

In [ ]:
reload_resources()

### 创建示例 prompt 和 skill

In [ ]:
(PROMPTS_DIR / "review.md").write_text(
    """请审查下面内容， 重点关注bug、可维护性和安全问题。
    {{input}}
    """,
    encoding="utf-8",
)

skill_dir = SKILLS_DIR / "project-summary"
skill_dir.mkdir(parents=True, exist_ok=True)

(skill_dir / "SKILL.md").write_text(
   """# Project Summary Skill

Use this skill when the user asks for a project overview.

Steps:
1. Inspect the directory structure.
2. Identify important files.
3. Read important documentation if present.
4. Summarize the project purpose and architecture.
""",
   encoding="utf-8",
)

In [ ]:
reload_resources()

### 加命令

In [ ]:
if "_base_handle_command_without_resources" not in globals():
    _base_handle_command_without_resources = handle_command


def show_resources():
    print(f"Settings: {SETTINGS_FILE}")
    print("\nAgents files:")
    for f in RESOURCES["agents_files"]:
        print(f"- {f['path']}")

    print("\nPrompts:")
    for name, p in RESOURCES["prompts"].items():
        print(f"- /{name}  ({p['path']})")

    print("\nSkills:")
    for name, s in RESOURCES["skills"].items():
        print(f"- /skill {name}  ({s['path']})")

In [ ]:
def submit_prompt(text, **kwargs):
    if SESSION_FILE is None:
        auto_resume_or_new()
    return ask(text, **kwargs)


def handle_command(text):
    parts = text.strip().split(maxsplit=2)
    cmd = parts[0][1:]
    arg1 = parts[1].strip() if len(parts) > 1 else ""
    arg2 = parts[2].strip() if len(parts) > 2 else ""

    if cmd in ("help", "?"):
        result = _base_handle_command_without_resources(text)
        print("""
        Resource commands:
        /reload                  Reload settings, AGENTS.md, prompts, skills
        /resources               Show loaded resources
        /settings                Show settings
        /settings save           Save current runtime settings
        /prompts                 List prompt templates
        /prompt NAME [TEXT]      Run prompt template
        /skills                  List skills
        /skill NAME [TEXT]       Run skill
        
        Prompt shortcuts:
        /review TEXT             If a prompt named review exists
        """.strip())
        return result
    
    if cmd == "reload":
        return reload_resources()

    if cmd == "resources":
        return show_resources()

    if cmd == "settings":
        if arg1 == "save":
            settings = default_settings()
            SETTINGS_FILE.write_text(
            json.dumps(settings, ensure_ascii=False, indent=2),
            encoding="utf-8",
            )
            print(f"Saved settings to {SETTINGS_FILE}")
            return

        print(json.dumps(RESOURCES["settings"], ensure_ascii=False, indent=2))
        return

    if cmd == "prompts":
        for name, p in RESOURCES["prompts"].items():
            print(f"- {name}: {p['path']}")
        return

    if cmd == "prompt":
        if not arg1:
            print("Usage: /prompt NAME [TEXT]")
            return
    
        expanded = expand_prompt_template(arg1, arg2)
        print(f"[prompt template: {arg1}]")
        return submit_prompt(expanded)

    if cmd == "skills":
        for name, s in RESOURCES["skills"].items():
            print(f"- {name}: {s['path']}")
        return

    if cmd == "skill":
        if not arg1:
            print("Usage: /skill NAME [TEXT]")
            return
    
        expanded = expand_skill(arg1, arg2)
        print(f"[skill: {arg1}]")
        return submit_prompt(expanded)

    # prompt shortcut: /review xxx
    if cmd in RESOURCES["prompts"]:
        expanded = expand_prompt_template(cmd, (arg1 + " " + arg2).strip())
        print(f"[prompt template: {cmd}]")
        return submit_prompt(expanded)

    return _base_handle_command_without_resources(text)

### 测试

In [369]:
run("/reloads")

<function __main__.handle_command(text)>

In [370]:
run('/prompts')

- review: D:\workflow\agent_harness\.mini-pi\prompts\review.md


## 升级 session,增加tree、fork、clone

In [373]:
import copy
from collections import defaultdict
from datetime import datetime, timezone

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def make_entry_id():
    return uuid.uuid4().hex[:8]


def set_session_cursor(entry_id):
    globals()["LAST_ENTRY_ID"] = entry_id
    globals()["CURRENT_ENTRY_ID"] = entry_id
    globals()["SESSION_CURSOR_ID"] = entry_id


def get_session_cursor():
    for name in ["LAST_ENTRY_ID", "CURRENT_ENTRY_ID", "SESSION_CURSOR_ID"]:
        value = globals().get(name)
        if value:
            return value
    return None


def append_session_entry(entry):
    global LAST_ENTRY_ID

    if SESSION_FILE is None:
        raise RuntimeError("No active session.")

    entry = dict(entry)
    entry.setdefault("id", make_entry_id())
    entry.setdefault("parentId", get_session_cursor())
    entry.setdefault("timestamp", now_iso())
    
    with open(SESSION_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    
    set_session_cursor(entry["id"])
    return entry


def save_message(message):
    return append_session_entry({
    "type": "message",
    "message": message,
    })

### 读取 session JSONL

In [374]:
def read_session_entries(path = None):
    path = Path(path or SESSION_FILE)

    entries = []
    if not path.exists():
        return entries

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        entries.append(json.loads(line))

    return entries

def session_id_entries(entries):
    return [e for e in entries if e.get("id")]

### 渲染 session tree

In [376]:
SESSION_TREE_INDEX = {}

def message_preview(message, max_len=80):
    role = message.get("role","?")
    content = message.get("content","")
    if isinstance(content, str):
        text = content
    elif isinstance(content,list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type")=="text":
                parts.append(block.get("text",""))
        text = " ".join(parts)
    else:
        text = str(content)
    text = " ".join(text.split())
    if len(text) > max_len:
        text = text[:max_len] + "..."
    return f"{role}: {text}"

def entry_preview(entry):
    typ = entry.get("type")
    
    if typ == "message":
        return message_preview(entry.get("message", {}))
    
    if typ == "compaction":
        return "compaction: " + entry.get("summary", "")[:80]
    
    if typ == "branch_summary":
        return "branch_summary: " + entry.get("summary", "")[:80]
    
    if typ == "model_change":
        return f"model_change: {entry.get('provider')}/{entry.get('model') or entry.get('modelId')}"
    
    return typ or "<unknown>"

def session_tree(show_tools=False):
    global SESSION_TREE_INDEX
    
    entries = session_id_entries(read_session_entries())
    ids = {e["id"] for e in entries}
    
    children = defaultdict(list)
    roots = []
    
    for e in entries:
        parent = e.get("parentId")
        if parent in ids:
            children[parent].append(e)
        else:
            roots.append(e)
    
    active = get_session_cursor()
    SESSION_TREE_INDEX = {}
    counter = {"n": 0}
    
    def visit(entry, prefix=""):
        msg = entry.get("message", {})
        role = msg.get("role")
        
        display_this = show_tools or role != "tool"
    
        if display_this:
            counter["n"] += 1
            idx = counter["n"]
            SESSION_TREE_INDEX[idx] = entry["id"]
    
            mark = " ← active" if entry["id"] == active else ""
            print(f"{idx:>3}. {prefix}{entry['id']} {entry_preview(entry)}{mark}")
            child_prefix = prefix + "  "
        else:
            child_prefix = prefix
    
        for child in children.get(entry["id"], []):
            visit(child, child_prefix)
    
    for root in roots:
        visit(root)
    
    if not SESSION_TREE_INDEX:
        print("<empty session tree>")

In [394]:
# session_tree()

### 根据 index/id 找 entry

In [378]:
def resolve_entry_ref(ref, entries=None):
    entries = session_id_entries(entries or read_session_entries())

    if ref is None or str(ref).strip() == "":
        current = get_session_cursor()
        if current:
            return current
        if entries:
            return entries[-1]["id"]
        raise ValueError("No session entries.")
    
    ref = str(ref).strip()
    
    if ref.isdigit():
        idx = int(ref)
        if idx in SESSION_TREE_INDEX:
            return SESSION_TREE_INDEX[idx]
    
    ids = [e["id"] for e in entries]
    
    exact = [x for x in ids if x == ref]
    if exact:
        return exact[0]
    
    prefix = [x for x in ids if x.startswith(ref)]
    if len(prefix) == 1:
        return prefix[0]
    
    if len(prefix) > 1:
        raise ValueError(f"Ambiguous entry id prefix: {ref} -> {prefix}")
    
    raise ValueError(f"Unknown entry ref: {ref}")

### 从root 到某个节点的路径

In [379]:
def path_to_entry(target_id, entries=None):
    entries = session_id_entries(entries or read_session_entries())
    by_id = {e["id"]: e for e in entries}
    
    if target_id not in by_id:
        raise ValueError(f"Unknown entry id: {target_id}")
    
    path = []
    cur = by_id[target_id]
    seen = set()
    
    while cur:
        eid = cur["id"]
        if eid in seen:
            raise RuntimeError("Cycle detected in session tree.")
        seen.add(eid)
    
        path.append(cur)
        parent = cur.get("parentId")
        cur = by_id.get(parent)
    
    path.reverse()
    return path

### 从路径恢复 messages

In [380]:
def base_system_message():
    if messages and messages[0].get("role") == "system":
        return copy.deepcopy(messages[0])
    
    if "rebuild_system_prompt" in globals():
        return {
        "role": "system",
        "content": rebuild_system_prompt(),
        }
    
    return {
    "role": "system",
    "content": "You are a minimal coding agent.",
    }


def messages_from_entry_path(path_entries):
    restored = [base_system_message()]

    for e in path_entries:
        typ = e.get("type")

        if typ == "message":
            restored.append(copy.deepcopy(e["message"]))

        elif typ == "compaction":
            restored.append({
            "role": "system",
            "content": "[Previous conversation summary]\n" +
            e.get("summary", ""),
            })
        
        elif typ == "branch_summary":
            restored.append({
            "role": "system",
            "content": "[Branch summary]\n" + e.get("summary", ""),
            })
    
    return restored

### 在当前 session 内跳转: tree jump

In [562]:
def jump_to_entry(ref):
    global messages

    entries = read_session_entries()
    target_id = resolve_entry_ref(ref, entries)
    path_entries = path_to_entry(target_id, entries)

    candidate_messages = messages_from_entry_path(path_entries)

    # 不允许跳转到尚未完成工具响应的节点
    assert_complete_tool_history(candidate_messages)

    messages = candidate_messages
    set_session_cursor(target_id)

    print(f"Jumped to entry: {target_id}")
    print(f"Restored messages: {len(messages)}")

    return target_id

In [395]:
# session_tree()

In [396]:
# jump_to_entry(3)

### Fork: 复制某个节点到新 session

In [564]:
def fork_session_at(ref=None, name=None):
    global SESSION_FILE, SESSION_ID, messages

    source_file = Path(SESSION_FILE)
    source_entries = read_session_entries(source_file)

    target_id = resolve_entry_ref(ref, source_entries)
    path_entries = path_to_entry(target_id, source_entries)

    candidate_messages = messages_from_entry_path(path_entries)

    # 在创建新 session 文件前检查
    assert_complete_tool_history(candidate_messages)
    

    session_dir = globals().get("SESSION_DIR", ROOT / ".mini-pi" /
    "sessions")
    session_dir.mkdir(parents=True, exist_ok=True)
    
    SESSION_ID = str(uuid.uuid4())
    filename = datetime.now().strftime("%Y%m%d-%H%M%S") + f"-fork-{SESSION_ID[:8]}.jsonl"
    SESSION_FILE = session_dir / filename
    
    header = {
    "type": "session",
    "version": 3,
    "id": SESSION_ID,
    "timestamp": now_iso(),
    "cwd": str(ROOT),
    "parentSession": str(source_file),
    }

    if name:
        header["name"] = name

    copied_entries = []
    parent_id = None

    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(json.dumps(header, ensure_ascii=False)+"\n")

        for old in path_entries:
            new = copy.deepcopy(old)
            new["id"] = make_entry_id()
            new["parentId"] = parent_id
            new["timestamp"] = now_iso()

            f.write(json.dumps(new,ensure_ascii=False)+"\n")

            parent_id = new["id"]
            copied_entries.append(new)

    set_session_cursor(parent_id)
    messages = candidate_messages

    print(f"Forked session:")
    print(f"source: {source_file}")
    print(f"target: {SESSION_FILE}")
    print(f"entries: {len(copied_entries)}")
    
    return SESSION_FILE

In [397]:
# session_tree()
# fork_session_at(6, name="eval-review-branch")

### Clone 当前分支

In [388]:
def clone_session(name=None):
    return fork_session_at(get_session_cursor(), name=name)

In [398]:
# clone_session("backup-before-refactor")

### 实验构建 独立内核

In [572]:
new_session()

New session: D:\workflow\agent_harness\.mini-pi\sessions\20260730-150345-a8fb5405.jsonl


WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260730-150345-a8fb5405.jsonl')

In [571]:
run("阅读并理解'mini_harness.ipynb',我要如何才能切换到deepseek-v4-pro?")

**Tool call:** `read_file`

```text
1: {
2:  "cells": [
3:   {
4:    "cell_type": "markdown",
5:    "metadata": {},
6:    "source": []
7:   },
8:   {
9:    "cell_type": "markdown",
10:    "metadata": {},
11:    "source": [
12:     "## End-to-end mini harness framework"
13:    ]
14:   },
15:   {
16:    "cell_type": "markdown",
17:    "metadata": {},
18:    "source": [
19:     "### First phase: minimum component"
20:    ]
21:   },
22:   {
23:    "cell_type": "code",
24:    "execution_count": 401,
25:    "metadata": {},
26:    "outputs": [],
27:    "source": [
28:     "# initialization\n",
29:     "import json\n",
30:     "from pathlib import Path\n",
31:     "from openai import OpenAI\n",
32:     "from IPython.display import display, Markdown\n",
33:     "import os\n",
34:     "from pprint import pprint\n",
35:     "\n",
36:     "client = OpenAI(\n",
37:     "    api_key=os.environ.get('DEEPSEEK_API_KEY'),\n",
38:     "    base_url=\"https://api.deepseek.com\")\n",
39:     "ROOT = Path.cwd()\n",
40:     "MODEL = \"deepseek-v4-flash\""
41:    ]
42:   },
43:   {
44:    "cell_type": "markdown",
45:    "metadata": {},
46:    "source": [
47:     "#### message container"
48:    ]
49:   },
50:   {
51:    "cell_type": "code",
52:    "execution_count": 402,
53:    "metadata": {},
54:    "outputs": [],
55:    "source": [
56:     "messages = [\n",
57:     "    {\n",
58:     "        \"role\":\"system\",\n",
59:     "        \"content\":f\"\"\"\n",
60:     "        You are a minimal coding agent running inside Jupyter Notebook.\n",
61:     "\n",
62:     "        Workspace root:\n",
63:     "        {ROOT}\n",
64:     "\n",
65:     "        You can use tools when needed.\n",
66:     "        Be concise.\n",
67:     "        \"\"\".strip()\n",
68:     "    }\n",
69:     "]"
70:    ]
71:   },
72:   {
73:    "cell_type": "code",
74:    "execution_count": 403,
75:    "metadata": {},
76:    "outputs": [
77:     {
78:      "name": "stdout",
79:      "output_type": "stream",
80:      "text": [
81:       "[{'content': 'You are a minimal coding agent running inside Jupyter Notebook.\\n'\n",
82:       "             '\\n'\n",
83:       "             '        Workspace root:\\n'\n",
84:       "             '        D:\\\\workflow\\\\agent_harness\\n'\n",
85:       "             '\\n'\n",
86:       "             '        You can use tools when needed.\\n'\n",
87:       "             '        Be concise.',\n",
88:       "  'role': 'system'}]\n"
89:      ]
90:     }
91:    ],
92:    "source": [
93:     "pprint(messages)"
94:    ]
95:   },
96:   {
97:    "cell_type": "markdown",
98:    "metadata": {},
99:    "source": [
100:     "#### Define one tool"
101:    ]
102:   },
103:   {
104:    "cell_type": "code",
105:    "execution_count": 404,
106:    "metadata": {},
107:    "outputs": [],
108:    "source": [
109:     "# a tool listing the directory\n",
110:     "def list_files(path=\".\"):\n",
111:     "    p = (ROOT/path).resolve()\n",
112:     "\n",
113:     "    if not p.is_relative_to(ROOT):\n",
114:     "     
```

**Tool call:** `read_file`

```text
200:   },
201:   {
202:    "cell_type": "code",
203:    "execution_count": 406,
204:    "metadata": {},
205:    "outputs": [],
206:    "source": [
207:     "def ask(prompt, max_turns=50):\n",
208:     "   messages.append({\n",
209:     "       \"role\": \"user\",\n",
210:     "       \"content\": prompt,\n",
211:     "   })\n",
212:     "\n",
213:     "   for turn in range(max_turns):\n",
214:     "       assistant_msg = call_model()\n",
215:     "       messages.append(assistant_msg)\n",
216:     "\n",
217:     "       content = assistant_msg.get(\"content\")\n",
218:     "       if content:\n",
219:     "           display(Markdown(content))\n",
220:     "\n",
221:     "       tool_calls = assistant_msg.get(\"tool_calls\") or []\n",
222:     "\n",
223:     "       if not tool_calls:\n",
224:     "           return assistant_msg\n",
225:     "\n",
226:     "       for tool_call in tool_calls:\n",
227:     "           name = tool_call[\"function\"][\"name\"]\n",
228:     "\n",
229:     "           display(Markdown(f\"**Tool call:** `{name}`\"))\n",
230:     "\n",
231:     "           result = execute_tool_call(tool_call)\n",
232:     "\n",
233:     "           display(Markdown(f\"```text\\n{result}\\n```\"))\n",
234:     "\n",
235:     "           messages.append({\n",
236:     "               \"role\": \"tool\",\n",
237:     "               \"tool_call_id\": tool_call[\"id\"],\n",
238:     "               \"content\": result,\n",
239:     "           })\n",
240:     "\n",
241:     "   raise RuntimeError(\"Agent exceeded max_turns\")"
242:    ]
243:   },
244:   {
245:    "cell_type": "markdown",
246:    "metadata": {},
247:    "source": [
248:     "### End-to-end test"
249:    ]
250:   },
251:   {
252:    "cell_type": "code",
253:    "execution_count": 407,
254:    "metadata": {},
255:    "outputs": [],
256:    "source": [
257:     "# ask(\"请查看当前目录，并告诉我这里大概有什么。\")"
258:    ]
259:   },
260:   {
261:    "cell_type": "markdown",
262:    "metadata": {},
263:    "source": [
264:     "## Add read_file"
265:    ]
266:   },
267:   {
268:    "cell_type": "markdown",
269:    "metadata": {},
270:    "source": [
271:     "### Add path resolver"
272:    ]
273:   },
274:   {
275:    "cell_type": "code",
276:    "execution_count": 408,
277:    "metadata": {},
278:    "outputs": [],
279:    "source": [
280:     "#add a path resolve avoid tools work outside of the workspace\n",
281:     "def resolve_workspace_path(path=\".\"):\n",
282:     "   raw = str(path).removeprefix(\"@\")\n",
283:     "   p = Path(raw)\n",
284:     "\n",
285:     "   if not p.is_absolute():\n",
286:     "       p = ROOT / p\n",
287:     "\n",
288:     "   p = p.resolve()\n",
289:     "   root = ROOT.resolve()\n",
290:     "\n",
291:     "   try:\n",
292:     "       p.relative_to(root)\n",
293:     "   except ValueError:\n",
294:     "       raise ValueError(f\"path escapes workspace: {path}\")\n",
295:     "\n",
296:     "   return p"
297:    ]
298:   },
299:   {
300:    "cell_type": "mar
```

**Tool call:** `read_file`

...

```text
$ python -c "import json; nb=json.load(open('mini_harness.ipynb','r',encoding='utf-8')); [print(f'cell {i}: {c[\"source\"]}') for i,c in enumerate(nb['cells']) if any('call_model' in line or 'MODEL' in line for line in c.get('source',[]))]"
       exit_code: 0

       STDOUT:
       cell 3: ['# initialization\n', 'import json\n', 'from pathlib import Path\n', 'from openai import OpenAI\n', 'from IPython.display import display, Markdown\n', 'import os\n', 'from pprint import pprint\n', '\n', 'client = OpenAI(\n', "    api_key=os.environ.get('DEEPSEEK_API_KEY'),\n", '    base_url="https://api.deepseek.com")\n', 'ROOT = Path.cwd()\n', 'MODEL = "deepseek-v4-flash"']
cell 13: ['def ask(prompt, max_turns=50):\n', '   messages.append({\n', '       "role": "user",\n', '       "content": prompt,\n', '   })\n', '\n', '   for turn in range(max_turns):\n', '       assistant_msg = call_model()\n', '       messages.append(assistant_msg)\n', '\n', '       content = assistant_msg.get("content")\n', '       if content:\n', '           display(Markdown(content))\n', '\n', '       tool_calls = assistant_msg.get("tool_calls") or []\n', '\n', '       if not tool_calls:\n', '           return assistant_msg\n', '\n', '       for tool_call in tool_calls:\n', '           name = tool_call["function"]["name"]\n', '\n', '           display(Markdown(f"**Tool call:** `{name}`"))\n', '\n', '           result = execute_tool_call(tool_call)\n', '\n', '           display(Markdown(f"```text\\n{result}\\n```"))\n', '\n', '           messages.append({\n', '               "role": "tool",\n', '               "tool_call_id": tool_call["id"],\n', '               "content": result,\n', '           })\n', '\n', '   raise RuntimeError("Agent exceeded max_turns")']
cell 104: ['def ask(prompt, max_turns = 50):\n', '    if SESSION_FILE is None:\n', '        raise RuntimeError("No active session. Run new_session() or resume_session(path) first.")\n', '    user_msg = {\n', '    "role":"user",\n', '    "content":prompt,\n', '    }\n', '    messages.append(user_msg)\n', '    save_message(user_msg)\n', '    for turn in range(max_turns):\n', '        assistant_msg = call_model()\n', '        messages.append(assistant_msg)\n', '        save_message(assistant_msg)\n', '        content = assistant_msg.get("content")\n', '        if content:\n', '            display(Markdown(content))\n', '        tool_calls = assistant_msg.get("tool_calls") or []\n', '\n', '        if not tool_calls:\n', '            return assistant_msg\n', '        for tool_call in tool_calls:\n', '            name = tool_call["function"]["name"]\n', '            display(Markdown(f"**Tool call:** `{name}`"))\n', '\n', '            result = execute_tool_call(tool_call)\n', '\n', '            display(Markdown(f"```text\\n{result[:3000]}\\n```"))\n', '\n', '            tool_msg = {\n', '                "role":"tool",\n', '                "tool_call_id":tool_call["id"],\n', '                "content":result,\n', '            }\n', '    
```

**Tool call:** `bash`

```text
$ python -c "import json; nb=json.load(open('mini_harness.ipynb','r',encoding='utf-8')); [print(f'cell {i}:', ''.join(c['source'])[:500]) for i,c in enumerate(nb['cells']) if any('def call_model' in line for line in c.get('source',[]))]"
       exit_code: 0

       STDOUT:
       cell 231: def call_model():
    kwargs = {
       "model": MODEL,
       "messages": messages,
       "tools": tools,
       "tool_choice": "auto",
    }
    
    if THINKING_LEVEL != "off" and model_supports_thinking(MODEL):
        kwargs["reasoning_effort"] = THINKING_LEVEL
    
    try:
        response = client.chat.completions.create(**kwargs)
    except Exception as e:
       # 如果模型不接受 reasoning_effort，自动降级重试
        if "reasoning_effort" in str(e) and "reasoning_effort" in kwargs:
            prin
cell 251: def call_model():
    llm = get_llm_client()
    kwargs = build_model_kwargs()

    try:
        response = llm.chat.completions.create(**kwargs)
    except Exception as e:
        msg = str(e)

        # 某些 DeepSeek 模型不接受 reasoning_effort / thinking 参数
        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
        
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_
cell 270: def call_model_streaming():
    llm = get_llm_client()
    kwargs = build_model_kwargs()
    kwargs["stream"] = True

    try:
        stream = llm.chat.completions.create(**kwargs)
    
    except Exception as e:
        msg = str(e)

        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
            
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_body", None)
 
cell 272: def call_model_agent():
    if not STREAMING_ENABLED:
        return call_model(), False
    
    try:
        return call_model_streaming(), True
    
    except Exception as e:
        print(f"\n[streaming warning] {type(e).__name__}: {e}")
        print("[streaming warning] fallback to non-streaming call")
        return call_model(), False
cell 293: if "_call_model_without_retry" not in globals():
    _call_model_without_retry = call_model

    def call_model():
        provider = globals().get("PROVIDER", "unknown")
        model = globals().get("MODEL", "unknown")
        return call_with_retry(
            _call_model_without_retry,
            label=f"{provider}/{model}:non_streaming",
        )

if "call_model_streaming" in globals():
    if "_call_model_streaming_without_retry" not in globals():
        _call_model_streaming_without_r


       STDERR:
       
       
```

我已经完整阅读了 `mini_harness.ipynb`。以下是分析和切换方案：

---

## 当前架构概述

这个 notebook 是一个**最小化 AI 编码 Agent**（类似 Pi Agent 的 harness），核心组件：

| 组件 | 说明 |
|------|------|
| **LLM 客户端** | OpenAI 兼容客户端 → `https://api.deepseek.com` |
| **当前模型** | `deepseek-v4-flash`（第40行 cell 3） |
| **工具** | `list_files`, `read_file`, `write_file`, `edit_file`, `bash` |
| **Agent Loop** | `ask()` 函数：用户提问 → call_model → 工具调用 → 循环 |
| **会话管理** | 后半部分引入了 session/模型切换/thinking 等高级功能 |

---

## 切换到 `deepseek-v4-pro` 的方法

有 **三种方式**，按推荐程度排序：

### 方式一：直接修改初始化 cell（最简单）

在 **cell 3**（第27-41行）中，将：

```python
MODEL = "deepseek-v4-flash"
```

改为：

```python
MODEL = "deepseek-v4-pro"
```

然后**从头重新运行所有 cell**。

### 方式二：使用 `/model` 命令（交互式）

如果已经运行了笔记本后半部分（包含 `handle_command` 的 session 系统），直接在 cell 中输入：

```
/model deepseek-v4-pro
```

这会调用内部的 `set_model("deepseek-v4-pro")`。

### 方式三：直接调用 `set_model()` 函数

在一个新的 code cell 中执行：

```python
set_model("deepseek-v4-pro")
```

---

## 注意事项

1. **API Key**：确保环境变量 `DEEPSEEK_API_KEY` 已正确设置，因为客户端在初始化时就读取了它（第37行）。

2. **Thinking / Reasoning**：`deepseek-v4-pro` 在 `MODEL_REGISTRY` 中标记为 `"reasoning": True`（cell 223），支持 reasoning 功能。但默认 `THINKING_LEVEL = "off"`，如果需要启用可以设置：
   ```python
   set_thinking("medium")  # 可选: off/minimal/low/medium/high
   ```

3. **价格**：`deepseek-v4-pro` 比 `deepseek-v4-flash` 更贵但推理能力更强，按需选择。

---

**最简单推荐**：直接改 cell 3 那行 `MODEL = "deepseek-v4-pro"`，然后重新运行全 notebook。需要我帮你直接修改吗？

{'content': '我已经完整阅读了 `mini_harness.ipynb`。以下是分析和切换方案：\n\n---\n\n## 当前架构概述\n\n这个 notebook 是一个**最小化 AI 编码 Agent**（类似 Pi Agent 的 harness），核心组件：\n\n| 组件 | 说明 |\n|------|------|\n| **LLM 客户端** | OpenAI 兼容客户端 → `https://api.deepseek.com` |\n| **当前模型** | `deepseek-v4-flash`（第40行 cell 3） |\n| **工具** | `list_files`, `read_file`, `write_file`, `edit_file`, `bash` |\n| **Agent Loop** | `ask()` 函数：用户提问 → call_model → 工具调用 → 循环 |\n| **会话管理** | 后半部分引入了 session/模型切换/thinking 等高级功能 |\n\n---\n\n## 切换到 `deepseek-v4-pro` 的方法\n\n有 **三种方式**，按推荐程度排序：\n\n### 方式一：直接修改初始化 cell（最简单）\n\n在 **cell 3**（第27-41行）中，将：\n\n```python\nMODEL = "deepseek-v4-flash"\n```\n\n改为：\n\n```python\nMODEL = "deepseek-v4-pro"\n```\n\n然后**从头重新运行所有 cell**。\n\n### 方式二：使用 `/model` 命令（交互式）\n\n如果已经运行了笔记本后半部分（包含 `handle_command` 的 session 系统），直接在 cell 中输入：\n\n```\n/model deepseek-v4-pro\n```\n\n这会调用内部的 `set_model("deepseek-v4-pro")`。\n\n### 方式三：直接调用 `set_model()` 函数\n\n在一个新的 code cell 中执行：\n\n```python\nset_model("deepseek-v4-pro")\n```\n\

In [573]:
run("/model deepseek-v4-pro")

Model set to: deepseek/deepseek-v4-pro
Thinking: high


## Event bus

In [578]:
from datetime import datetime, timezone
import traceback

EVENT_SUBSCRIBERS = []
EVENT_LOG = []
EVENT_LOG_ENABLED = True

In [581]:
def event_now():
    return datetime.now(timezone.utc).isoformat()

def emit_event(event):
    event = dict(event)
    event.setdefault("timestamp", event_now())

    if EVENT_LOG_ENABLED:
        EVENT_LOG.append(event)

    for subscriber in list(EVENT_SUBSCRIBERS):
        try:
            subscriber(event)
        except Exception as e:
            print(f"[event subscriber error] {type(e).__name__}: {e}")

    return event


def subscribe_events(handler):
    EVENT_SUBSCRIBERS.append(handler)

    def unsubscribe():
        if handler in EVENT_SUBSCRIBERS:
            EVENT_SUBSCRIBERS.remove(handler)

    return unsubscribe


def clear_event_log():
    EVENT_LOG.clear()


def event_log_tail(n=20):
    for e in EVENT_LOG[-n:]:
        print(json.dumps(e, ensure_ascii=False, indent=2, default=str))

### Debug 监听器

In [582]:
EVENT_DEBUG_UNSUBSCRIBE = None


def debug_event_printer(event):
    typ = event.get("type")

    # streaming 文本增量太多，默认跳过
    if typ == "message_update":
        return

    if typ == "tool_execution_start":
        print(f"[event] tool_start: {event.get('toolName')} {event.get('args')}")
        return

    if typ == "tool_execution_end":
        print(f"[event] tool_end: {event.get('toolName')} error={event.get('isError')}")
        return

    print(f"[event] {typ}")


def event_debug(on=True):
    global EVENT_DEBUG_UNSUBSCRIBE

    if on:
        if EVENT_DEBUG_UNSUBSCRIBE is None:
            EVENT_DEBUG_UNSUBSCRIBE = subscribe_events(debug_event_printer)
        print("Event debug enabled.")
    else:
        if EVENT_DEBUG_UNSUBSCRIBE is not None:
            EVENT_DEBUG_UNSUBSCRIBE()
            EVENT_DEBUG_UNSUBSCRIBE = None
        print("Event debug disabled.")

In [583]:
event_debug(True)

Event debug enabled.


### 包装工具执行事件
所有工具调用产生：
tool_execution_start
too_execution_end

In [585]:
def parse_tool_call_args_for_event(tool_call):
    raw = tool_call.get("function", {}).get("arguments", "{}")

    if isinstance(raw, dict):
        return raw

    try:
        return json.loads(raw or "{}")
    except Exception:
        return {"_raw": raw}


def short_text_for_event(value, max_chars=1000):
    text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + f"\n...[truncated {len(text)} chars]"
    return text


if "_execute_tool_call_without_events" not in globals():
    _execute_tool_call_without_events = execute_tool_call


def execute_tool_call(tool_call):
    tool_call_id = tool_call.get("id")
    tool_name = tool_call.get("function", {}).get("name", "")
    args = parse_tool_call_args_for_event(tool_call)
    
    emit_event({
    "type": "tool_execution_start",
    "toolCallId": tool_call_id,
    "toolName": tool_name,
    "args": args,
    })

    try:
        result = _execute_tool_call_without_events(tool_call)
        is_error = isinstance(result, str) and result.startswith("ERROR")
    
        emit_event({
        "type": "tool_execution_end",
        "toolCallId": tool_call_id,
        "toolName": tool_name,
        "resultPreview": short_text_for_event(result),
        "isError": is_error,
        })
        
        return result

    except Exception as e:
        result = f"ERROR: {type(e).__name__}: {e}"

        emit_event({
        "type": "tool_execution_end",
        "toolCallId": tool_call_id,
        "toolName": tool_name,
        "resultPreview": result,
        "isError": True,
        "traceback": traceback.format_exc(),
        })
    
    return result

### 替换 streaming 模型调用， 增加 message_update

In [587]:
def call_model_streaming_core_evented():
    llm = get_llm_client()
    kwargs = build_model_kwargs()
    kwargs["stream"] = True
    
    try:
        stream = llm.chat.completions.create(**kwargs)
    
    except Exception as e:
        msg = str(e)
    
        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
            
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_body", None)
            
            stream = llm.chat.completions.create(**kwargs)
        else:
            raise
    
    role = "assistant"
    content_parts = []
    tool_calls_by_index = {}

    for chunk in stream:
        if not chunk.choices:
            continue
    
        choice = chunk.choices[0]
        delta = choice.delta
        
        if delta is None:
            continue
    
        d = delta.model_dump(exclude_none=True)
    
        if d.get("role"):
            role = d["role"]
    
        if d.get("content"):
            text = d["content"]
            content_parts.append(text)
            
            emit_event({
            "type": "message_update",
            "assistantMessageEvent": {
            "type": "text_delta",
            "delta": text,
            },
            "delta": text,
            })
        
            print(text, end="", flush=True)
        
        for tc in d.get("tool_calls") or []:
            merge_tool_call_delta(tool_calls_by_index, tc)
    
    content = "".join(content_parts)
    
    if content:
        print()
    
    assistant_msg = {
    "role": role,
    "content": content if content else None,
    }
    
    tool_calls = finalize_streamed_tool_calls(tool_calls_by_index)
    
    if tool_calls:
        assistant_msg["tool_calls"] = tool_calls
    
    return assistant_msg
    
    
def call_model_streaming():
    provider = globals().get("PROVIDER", "unknown")
    model = globals().get("MODEL", "unknown")
    
    if "call_with_retry" in globals():
        return call_with_retry(
        call_model_streaming_core_evented,
        label=f"{provider}/{model}:streaming",
        )
    
    return call_model_streaming_core_evented()

### 统一模型调用入口

In [590]:
def call_model_agent():
    if not globals().get("STREAMING_ENABLED", True):
        return call_model(), False
    
    try:
        return call_model_streaming(), True
    
    except Exception as e:
        print(f"\n[streaming warning] {type(e).__name__}: {e}")
        print("[streaming warning] fallback to non-streaming call")
        return call_model(), False

### 替换 ask(), 加入 agent/turn/message 事件

In [642]:
def ask(prompt, max_turns=5000):
    if SESSION_FILE is None:
        raise RuntimeError("No active session. Run new_session() or resume_session(path) first.")

    assert_complete_tool_history(messages)
    
    emit_event({
    "type": "agent_start",
    "promptPreview": short_text_for_event(prompt, 500),
    })
    
    try:
        user_msg = {
        "role": "user",
        "content": prompt,
        }
        
        messages.append(user_msg)
        save_message(user_msg)
        
        for turn in range(max_turns):
            emit_event({
            "type": "turn_start",
            "turn": turn + 1,
            })
            
            maybe_auto_compact(reason="before_model_call")
            
            emit_event({
            "type": "message_start",
            "role": "assistant",
            })
            
            assistant_msg, already_rendered = call_model_agent()
            
            messages.append(assistant_msg)
            save_message(assistant_msg)
    
            emit_event({
            "type": "message_end",
            "message": assistant_msg,
            })
    
            content = assistant_msg.get("content")
            if content and not already_rendered:
                display(Markdown(content))
    
            tool_calls = assistant_msg.get("tool_calls") or []
    
            if not tool_calls:
                emit_event({
                "type": "turn_end",
                "turn": turn + 1,
                "toolResults": [],
                })
        
                emit_event({
                "type": "agent_end",
                "success": True,
                })
        
                return assistant_msg
    
            tool_results = []
    
            for tool_call in tool_calls:
                name = tool_call["function"]["name"]
                
                display(Markdown(f"**Tool call:** `{name}`"))
                
                result = execute_tool_call(tool_call)
                
                display(Markdown(f"```text\n{str(result)[:3000]}\n```"))
                
                tool_msg = {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": str(result),
                }
                
                messages.append(tool_msg)
                save_message(tool_msg)
                tool_results.append(tool_msg)
                
            emit_event({
            "type": "turn_end",
            "turn": turn + 1,
            "toolResults": tool_results,
            })
            
        raise RuntimeError("Agent exceeded max_turns")
    
    except Exception as e:
        emit_event({
        "type": "agent_end",
        "success": False,
        "error": f"{type(e).__name__}: {e}",
        })
        raise

### 测试

In [643]:
event_debug(True)

Event debug enabled.


In [593]:
run("hello, 用一句化回答")

[event] agent_start
[event] turn_start
[event] message_start
[event] message_end


你好！`agent_core_evol.ipynb` 已就绪，Agent 的本质就是：**把工具调用结果塞回对话，让模型在循环中反复思考直到给出最终答案。**

[event] turn_end
[event] agent_end


{'content': '你好！`agent_core_evol.ipynb` 已就绪，Agent 的本质就是：**把工具调用结果塞回对话，让模型在循环中反复思考直到给出最终答案。**',
 'role': 'assistant',
 'reasoning_content': 'The user is saying hello and asking me to respond with just one sentence.'}

In [594]:
run("查看当前目录，并简单总结")

[event] agent_start
[event] turn_start
[event] message_start
[event] message_end


**Tool call:** `list_files`

[event] tool_start: list_files {'path': '.'}
[event] tool_end: list_files error=False


```text
.agents/
.git/
.gitignore
.ipynb_checkpoints/
.mini-pi/
.nb-agent/
agent_core.ipynb
course/
courses/
harness_lab/
mini_harness.ipynb
notes/
Omega
omega.py
README.txt
```

[event] turn_end
[event] turn_start
[event] message_start
[event] message_end


当前目录主要包含：

- **`mini_harness.ipynb`** — 原始的全功能 Agent harness，渐进式构建（383 个 cell）
- **`agent_core_evol.ipynb`** — 刚才创建的极简教学版，聚焦核心循环
- **`harness_lab/`**、**`course/`**、**`courses/`** — 课程/实验相关代码
- **`Omega`、`omega.py`** — 另一个 agent 实现
- **`notes/`** — 笔记
- **`.agents/`、`.mini-pi/`、`.nb-agent/`** — agent 运行时的内部目录
- **`README.txt`** — 项目说明

一句话：这是一个 **Agent harness 学习项目**，从 `mini_harness.ipynb` 逐步演化，配合课程资料和多个实验实现。

[event] turn_end
[event] agent_end


{'content': '当前目录主要包含：\n\n- **`mini_harness.ipynb`** — 原始的全功能 Agent harness，渐进式构建（383 个 cell）\n- **`agent_core_evol.ipynb`** — 刚才创建的极简教学版，聚焦核心循环\n- **`harness_lab/`**、**`course/`**、**`courses/`** — 课程/实验相关代码\n- **`Omega`、`omega.py`** — 另一个 agent 实现\n- **`notes/`** — 笔记\n- **`.agents/`、`.mini-pi/`、`.nb-agent/`** — agent 运行时的内部目录\n- **`README.txt`** — 项目说明\n\n一句话：这是一个 **Agent harness 学习项目**，从 `mini_harness.ipynb` 逐步演化，配合课程资料和多个实验实现。',
 'role': 'assistant',
 'reasoning_content': "The user wants a simple summary of what's in the directory."}

## Export / Import 

### 基础helper

In [644]:
from pathlib import Path
import json
import shutil
import shlex
import uuid
import html as html_lib
from datetime import datetime, timezone

EXPORT_DIR = ROOT / ".mini-pi" / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def export_now_iso():
    return datetime.now(timezone.utc).isoformat()


def export_stamp():
    return datetime.now().strftime("%Y%m%d-%H%M%S")


def session_file_path(path=None):
    p = path or globals().get("SESSION_FILE")
    if p is None:
        raise RuntimeError("No active session.")
    return Path(p)


def read_jsonl_entries(path):
    path = Path(path)
    entries = []

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            entries.append(json.loads(line))
    
    return entries


def export_output_path(out, ext):
    if out:
        p = Path(out).expanduser()
        if not p.is_absolute():
            p = ROOT / p
        if not p.suffix:
            p = p.with_suffix("." + ext)
    else:
        sid = globals().get("SESSION_ID", "session")
        p = EXPORT_DIR / f"{export_stamp()}-{str(sid)[:8]}.{ext}"
    
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

### 获取当前 brach entries

In [645]:
def entry_id_items(entries):
    return [e for e in entries if e.get("id")]


def current_cursor_id():
    if "get_session_cursor" in globals():
        return get_session_cursor()
    
    for name in ["LAST_ENTRY_ID", "CURRENT_ENTRY_ID", "SESSION_CURSOR_ID"]:
        value = globals().get(name)
        if value:
            return value
    
    return None


def branch_entries_from_session(path=None, target_id=None):
    path = session_file_path(path)
    entries = read_jsonl_entries(path)
    items = entry_id_items(entries)

    if not items:
        return []

    if target_id is None:
        target_id = current_cursor_id() or items[-1]["id"]

    by_id = {e["id"]: e for e in items}

    if target_id not in by_id:
        target_id = items[-1]["id"]

    result = []
    seen = set()
    cur = by_id[target_id]

    while cur:
        eid = cur["id"]
        if eid in seen:
            raise RuntimeError("Cycle detected in session tree.")
        seen.add(eid)
        
        result.append(cur)
        parent = cur.get("parentId")
        cur = by_id.get(parent)
        
    result.reverse()
    return result

### Markdown / HTML 渲染

In [646]:
def content_to_text(content):
    if content is None:
        return ""

    if isinstance(content, str):
        return content
    
    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, dict):
                typ = block.get("type")

                if typ == "text":
                    parts.append(block.get("text", ""))

                elif typ == "image":
                    parts.append(f"[image: {block.get('mimeType','unknown')}]")

                elif typ == "thinking":
                    parts.append("[thinking]\n" + block.get("thinking", ""))

                elif typ == "toolCall":
                    parts.append(
                       f"[tool call: {block.get('name')} "
                       f"{json.dumps(block.get('arguments', {}),
                    ensure_ascii=False)}]"
                    )

                else:
                    parts.append(json.dumps(block, ensure_ascii=False))

            else:
                parts.append(str(block))

        return "\n".join(p for p in parts if p)

    return str(content)


def render_message_md(message, include_system=False):
    role = message.get("role", "?")

    if role == "system" and not include_system:
        return ""

    title = {
    "system": "System",
    "user": "User",
    "assistant": "Assistant",
    "tool": "Tool Result",
    "toolResult": "Tool Result",
    }.get(role, role)

    lines = [f"## {title}"]

    if role in ("tool", "toolResult"):
        tid = message.get("tool_call_id") or message.get("toolCallId")
        if tid:
            lines.append(f"tool_call_id: `{tid}`")

    text = content_to_text(message.get("content"))
    if text:
        lines.append(text)

    tool_calls = message.get("tool_calls") or []
    if tool_calls:
        lines.append("Tool calls:")
        lines.append("```json")
        lines.append(json.dumps(tool_calls, ensure_ascii=False, indent=2))
        lines.append("```")

    return "\n\n".join(lines)


def build_session_markdown(path=None, include_system=False):
    path = session_file_path(path)
    raw_entries = read_jsonl_entries(path)
    header = raw_entries[0] if raw_entries and raw_entries[0].get("type") =="session" else {}
    branch = branch_entries_from_session(path)

    lines = [
    "# Mini Pi Session Export",
    "",
    f"- exportedAt: `{export_now_iso()}`",
    f"- sessionFile: `{path}`",
    f"- sessionId: `{header.get('id', globals().get('SESSION_ID',
    'unknown'))}`",
    f"- cwd: `{header.get('cwd', ROOT)}`",
    "",
    "---",
    "",
    ]

    for entry in branch:
        typ = entry.get("type")

        if typ == "message":
            block = render_message_md(entry.get("message", {}), include_system=include_system)
            if block:
                lines.append(block)
                lines.append("\n---\n")

        elif typ == "compaction":
            lines.append("## Compaction Summary")
            lines.append(entry.get("summary", ""))
            lines.append("\n---\n")

        elif typ == "branch_summary":
            lines.append("## Branch Summary")
            lines.append(entry.get("summary", ""))
            lines.append("\n---\n")

    return "\n".join(lines)

### Export 实现

In [647]:
def export_session_md(out=None, include_system=False):
    target = export_output_path(out, "md")
    text = build_session_markdown(include_system=include_system)
    target.write_text(text, encoding="utf-8")
    print(f"Exported Markdown: {target}")
    return target


def export_session_html(out=None, include_system=False):
    target = export_output_path(out, "html")
    md = build_session_markdown(include_system=include_system)
    
    html_text = f"""<!doctype html>
    <html>
    <head>
    <meta charset="utf-8">
    <title>Mini Pi Session Export</title>
    <style>
    body {{
    font-family: system-ui, -apple-system, BlinkMacSystemFont, sans-serif;
    line-height: 1.5;
    max-width: 960px;
    margin: 40px auto;
    padding: 0 20px;
    }}
    pre {{
    white-space: pre-wrap;
    word-wrap: break-word;
    background: #f6f8fa;
    padding: 16px;
    border-radius: 8px;
    }}
    </style>
    </head>
    <body>
    <pre>{html_lib.escape(md)}</pre>
    </body>
    </html>
    """
    
    target.write_text(html_text, encoding="utf-8")
    print(f"Exported HTML: {target}")
    return target


def export_session_jsonl(out=None):
    source = session_file_path()
    target = export_output_path(out, "jsonl")
    shutil.copyfile(source, target)
    print(f"Exported JSONL: {target}")
    return target


def export_session(fmt="html", out=None, include_system=False):
    fmt = (fmt or "html").lower()
    
    if fmt in ("md", "markdown"):
        return export_session_md(out, include_system=include_system)
    
    if fmt == "html":
        return export_session_html(out, include_system=include_system)
    
    if fmt == "jsonl":
        return export_session_jsonl(out)
    
    raise ValueError("fmt must be one of: html, md, jsonl")

In [648]:
export_session("md")
export_session("html")
export_session("jsonl")

Exported Markdown: D:\workflow\agent_harness\.mini-pi\exports\20260731-012008-a8fb5405.md
Exported HTML: D:\workflow\agent_harness\.mini-pi\exports\20260731-012008-a8fb5405.html
Exported JSONL: D:\workflow\agent_harness\.mini-pi\exports\20260731-012008-a8fb5405.jsonl


WindowsPath('D:/workflow/agent_harness/.mini-pi/exports/20260731-012008-a8fb5405.jsonl')

### Import 实现

In [649]:
def set_cursor_for_import(entry_id):
    if "set_session_cursor" in globals():
        set_session_cursor(entry_id)
    else:
        globals()["LAST_ENTRY_ID"] = entry_id
        globals()["CURRENT_ENTRY_ID"] = entry_id
        globals()["SESSION_CURSOR_ID"] = entry_id


def runtime_messages_from_branch(branch):
    restored = []

    if "rebuild_system_prompt" in globals():
        restored.append({
        "role": "system",
        "content": rebuild_system_prompt(),
        })
    elif "messages" in globals() and messages and messages[0].get("role") =="system":
        restored.append(messages[0])
    else:
        restored.append({
        "role": "system",
        "content": "You are a minimal coding agent.",
        })

    for entry in branch:
        typ = entry.get("type")

        if typ == "message":
            msg = entry.get("message", {})
            role = msg.get("role")

            if role in ("system", "user", "assistant", "tool"):
               restored.append(msg)

            elif role == "toolResult":
               restored.append({
                   "role": "tool",
                   "tool_call_id": msg.get("toolCallId"),
                   "content": content_to_text(msg.get("content")),
               })

            else:
               restored.append({
                   "role": "user",
                   "content": f"[imported {role} message]\n{content_to_text(msg.get('content'))}",
               })

        elif typ == "compaction":
            restored.append({
               "role": "system",
               "content": "[Previous conversation summary]\n" +
            entry.get("summary", ""),
            })

        elif typ == "branch_summary":
            restored.append({
               "role": "system",
               "content": "[Branch summary]\n" + entry.get("summary", ""),
            })

    return restored


def import_session(path, copy_into_sessions=True):
    global SESSION_FILE, SESSION_ID, messages

    source = Path(path).expanduser()
    if not source.is_absolute():
        source = ROOT / source

    source = source.resolve()

    if not source.exists():
        raise FileNotFoundError(source)

    entries = read_jsonl_entries(source)
    if not entries:
        raise ValueError("Empty JSONL session file.")

    header = entries[0] if entries[0].get("type") == "session" else {}

    session_dir = globals().get("SESSION_DIR", ROOT / ".mini-pi" / "sessions")
    session_dir = Path(session_dir)
    session_dir.mkdir(parents=True, exist_ok=True)

    if copy_into_sessions:
        target = session_dir / f"{export_stamp()}-import-{source.stem}.jsonl"
        shutil.copyfile(source, target)
    else:
        target = source

    SESSION_FILE = target
    SESSION_ID = header.get("id") or str(uuid.uuid4())
    

    items = entry_id_items(entries)
    target_id = items[-1]["id"] if items else None
    
    branch = branch_entries_from_session(
        target,
        target_id=target_id,
    )
    
    messages = runtime_messages_from_branch(branch)

    # 导入后立即验证
    assert_complete_tool_history(messages)
    
    last_id = branch[-1]["id"] if branch else None
    if last_id:
        set_cursor_for_import(last_id)

    print(f"Imported session: {target}")
    print(f"SESSION_ID: {SESSION_ID}")
    print(f"Restored messages: {len(messages)}")
    
    return target

In [608]:
p = export_session("jsonl")

Exported JSONL: D:\workflow\agent_harness\.mini-pi\exports\20260731-001600-a8fb5405.jsonl


In [609]:
import_session(p)

Imported session: D:\workflow\agent_harness\.mini-pi\sessions\20260731-001608-import-20260731-001600-a8fb5405.jsonl
SESSION_ID: a8fb54050c864bd0901f7d33392acdad
Restored messages: 57


WindowsPath('D:/workflow/agent_harness/.mini-pi/sessions/20260731-001608-import-20260731-001600-a8fb5405.jsonl')

### 加 /export 和 /import 命令

In [650]:
if "_base_handle_command_without_export_import" not in globals():
    _base_handle_command_without_export_import = handle_command


def parse_export_args(args):
    parts = shlex.split(args)
    
    if not parts:
        return "html", None

    first = parts[0].lower()

    if first in ("html", "md", "markdown", "jsonl"):
        fmt = first
        out = parts[1] if len(parts) > 1 else None
        return fmt, out

    # 根据扩展名推断
    out = parts[0]
    suffix = Path(out).suffix.lower()

    if suffix == ".md":
        return "md", out
    if suffix == ".jsonl":
        return "jsonl", out
    return "html", out


def handle_command(text):
    body = text.strip()[1:]
    cmd, _, args = body.partition(" ")
    cmd = cmd.strip().lower()
    args = args.strip()

    if cmd == "export":
        fmt, out = parse_export_args(args)
        return export_session(fmt, out)

    if cmd == "import":
        if not args:
            print("Usage: /import FILE.jsonl")
            return

        parts = shlex.split(args)
        return import_session(parts[0])

    return _base_handle_command_without_export_import(text)

In [611]:
run("/export md")

Exported Markdown: D:\workflow\agent_harness\.mini-pi\exports\20260731-002016-a8fb5405.md


WindowsPath('D:/workflow/agent_harness/.mini-pi/exports/20260731-002016-a8fb5405.md')

In [612]:
run("/export html my-session.html")

Exported HTML: D:\workflow\agent_harness\my-session.html


WindowsPath('D:/workflow/agent_harness/my-session.html')

## token 消费计算

### 使用格式

In [651]:
LAST_PROVIDER_USAGE = None


def obj_to_dict(x):
    if x is None:
        return None
    if isinstance(x, dict):
        return x
    if hasattr(x, "model_dump"):
        return x.model_dump(exclude_none=True)
    if hasattr(x, "__dict__"):
        return dict(x.__dict__)
    return {"value": str(x)}


def normalize_provider_usage(usage):
    d = obj_to_dict(usage)
    if not d:
        return None

    input_tokens = (
    d.get("prompt_tokens")
    or d.get("input_tokens")
    or d.get("input")
    or 0
    )

    output_tokens = (
    d.get("completion_tokens")
    or d.get("output_tokens")
    or d.get("output")
    or 0
    )

    total_tokens = (
    d.get("total_tokens")
    or d.get("totalTokens")
    or int(input_tokens) + int(output_tokens)
    )

    prompt_details = d.get("prompt_tokens_details") or {}
    completion_details = d.get("completion_tokens_details") or {}

    cache_read = (
    d.get("cache_read")
    or d.get("cacheRead")
    or d.get("cached_tokens")
    or prompt_details.get("cached_tokens")
    or d.get("prompt_cache_hit_tokens")
    or 0
    )

    cache_write = (
    d.get("cache_write")
    or d.get("cacheWrite")
    or d.get("cache_creation_input_tokens")
    or 0
    )

    reasoning_tokens = (
    d.get("reasoning_tokens")
    or completion_details.get("reasoning_tokens")
    or 0
    )

    return {
    "input": int(input_tokens),
    "output": int(output_tokens),
    "cacheRead": int(cache_read),
    "cacheWrite": int(cache_write),
    "reasoning": int(reasoning_tokens),
    "totalTokens": int(total_tokens),
    "raw": d,
    }


def set_last_provider_usage(usage):
    global LAST_PROVIDER_USAGE
    LAST_PROVIDER_USAGE = normalize_provider_usage(usage)
    return LAST_PROVIDER_USAGE

### 计算 cost

In [652]:
def usage_cost_breakdown(usage, provider=None, model=None):
    provider = provider or globals().get("PROVIDER", "unknown")
    model = model or globals().get("MODEL", "unknown")
    
    info = get_model_accounting(provider, model)
    
    input_price = info.get("input_per_1m")
    output_price = info.get("output_per_1m")

    if input_price is None or output_price is None:
        return {
           "input": None,
           "output": None,
           "cacheRead": None,
           "cacheWrite": None,
           "total": None,
        }

    input_cost = usage["input"] / 1_000_000 * input_price
    output_cost = usage["output"] / 1_000_000 * output_price

    return {
    "input": input_cost,
    "output": output_cost,
    "cacheRead": 0.0,
    "cacheWrite": 0.0,
    "total": input_cost + output_cost,
    }


def make_pi_usage(usage, estimated=False):
    if usage is None:
        return None

    cost = usage_cost_breakdown(usage)

    return {
    "input": usage["input"],
    "output": usage["output"],
    "cacheRead": usage.get("cacheRead", 0),
    "cacheWrite": usage.get("cacheWrite", 0),
    "reasoning": usage.get("reasoning", 0),
    "totalTokens": usage["totalTokens"],
    "cost": cost,
    "estimated": estimated,
    "raw": usage.get("raw"),
    }


def make_estimated_pi_usage(input_tokens, output_tokens):
    usage = {
    "input": int(input_tokens),
    "output": int(output_tokens),
    "cacheRead": 0,
    "cacheWrite": 0,
    "reasoning": 0,
    "totalTokens": int(input_tokens) + int(output_tokens),
    "raw": None,
    }
    return make_pi_usage(usage, estimated=True)

### assistant message 注入 provider/model/usage

In [653]:
def normalize_stop_reason(reason):
    mapping = {
    "stop": "stop",
    "length": "length",
    "tool_calls": "toolUse",
    "tool_use": "toolUse",
    "content_filter": "error",
    }
    return mapping.get(reason, reason or "stop")


def annotate_assistant_message(message, usage=None, stop_reason=None):
    message = dict(message)
    
    message.setdefault("api", "chat.completions")
    message["provider"] = globals().get("PROVIDER", "unknown")
    message["model"] = globals().get("MODEL", "unknown")
    message["timestamp"] = event_now() if "event_now" in globals() else export_now_iso()
    message["stopReason"] = normalize_stop_reason(stop_reason)
    
    if usage is not None:
        message["usage"] = make_pi_usage(usage, estimated=False)
    
    return message

### 估计token

In [661]:
import math
import json

_TOKEN_ENCODER = globals().get("_TOKEN_ENCODER", None)


def get_token_encoder():
    global _TOKEN_ENCODER

    if _TOKEN_ENCODER is not None:
        return _TOKEN_ENCODER

    try:
        import tiktoken
        _TOKEN_ENCODER = tiktoken.get_encoding("cl100k_base")
    except Exception:
        _TOKEN_ENCODER = False

    return _TOKEN_ENCODER


def estimate_text_tokens(text):
    text = "" if text is None else str(text)

    enc = get_token_encoder()
    if enc:
        return len(enc.encode(text))

    # fallback：中文大约 1~1.2 token/字，英文大约 4 chars/token
    zh = sum(1 for ch in text if "\u4e00" <= ch <= "\u9fff")
    other = len(text) - zh

    return max(1, math.ceil(zh * 1.2 + other / 4))


if "content_to_text" not in globals():
    def content_to_text(content):
        if content is None:
            return ""

        if isinstance(content, str):
            return content

        if isinstance(content, list):
            parts = []
            for block in content:
                if isinstance(block, dict):
                    typ = block.get("type")
                    if typ == "text":
                        parts.append(block.get("text", ""))
                    elif typ == "image":
                        parts.append(f"[image: {block.get('mimeType','unknown')}]")
                    elif typ == "thinking":
                        parts.append(block.get("thinking", ""))
                    elif typ == "toolCall":
                        parts.append(json.dumps(block, ensure_ascii=False))
                    else:
                        parts.append(json.dumps(block, ensure_ascii=False))
                else:
                    parts.append(str(block))
            return "\n".join(parts)
    
        return str(content)


def message_accounting_text(message):
    parts = []
    
    parts.append(str(message.get("role", "")))
    parts.append(content_to_text(message.get("content", "")))
    
    if message.get("tool_calls"):
        parts.append(json.dumps(message["tool_calls"], ensure_ascii=False))
    
    if message.get("tool_call_id"):
        parts.append(str(message["tool_call_id"]))
    
    if message.get("usage"):
    # usage 不参与请求内容，通常不该计入 context
        pass
    
    return "\n".join(p for p in parts if p)


def estimate_message_tokens(message):
# role/message framing 给一点固定开销
    return 4 + estimate_text_tokens(message_accounting_text(message))


def estimate_messages_tokens(msgs):
    return 3 + sum(estimate_message_tokens(m) for m in msgs)



DEFAULT_CONTEXT_WINDOW = 128000

# 价格只是示例；如果要精确计费，请按 DeepSeek / OpenAI 官方当前价格修改
MODEL_ACCOUNTING = globals().get("MODEL_ACCOUNTING", {})

MODEL_ACCOUNTING.update({
("deepseek", "deepseek-v4-flash"): {
"context_window": 64000,
"input_per_1m": 0.27,
"output_per_1m": 1.10,
},
("deepseek", "deepseek-v4-pro"): {
"context_window": 64000,
"input_per_1m": 0.55,
"output_per_1m": 2.19,
},
("openai", "gpt-4.1"): {
"context_window": 1000000,
"input_per_1m": None,
"output_per_1m": None,
},
})


def get_model_accounting(provider=None, model=None):
    provider = provider or globals().get("PROVIDER", "unknown")
    model = model or globals().get("MODEL", "unknown")
    
    return MODEL_ACCOUNTING.get((provider, model), {})


def get_context_window(provider=None, model=None):
    provider = provider or globals().get("PROVIDER", "unknown")
    model = model or globals().get("MODEL", "unknown")
    
    info = get_model_accounting(provider, model)

    if info.get("context_window"):
        return int(info["context_window"])

# 如果 PROVIDER_REGISTRY 里写了 context_window，也可以从那里读
    try:
        cfg = PROVIDER_REGISTRY[provider]["models"][model]
        if cfg.get("context_window"):
            return int(cfg["context_window"])
    except Exception:
        pass

    return DEFAULT_CONTEXT_WINDOW


def estimate_cost_usd(input_tokens, output_tokens, provider=None, model=None):
    info = get_model_accounting(provider, model)
    
    input_price = info.get("input_per_1m")
    output_price = info.get("output_per_1m")
    
    if input_price is None or output_price is None:
        return None
    
    return (
    input_tokens / 1_000_000 * input_price
    + output_tokens / 1_000_000 * output_price
    )


def usage_cost_breakdown(usage, provider=None, model=None):
    provider = provider or globals().get("PROVIDER", "unknown")
    model = model or globals().get("MODEL", "unknown")
    
    info = get_model_accounting(provider, model)
    
    input_price = info.get("input_per_1m")
    output_price = info.get("output_per_1m")
    
    if input_price is None or output_price is None:
        return {
        "input": None,
        "output": None,
        "cacheRead": None,
        "cacheWrite": None,
        "total": None,
        }

    input_cost = usage.get("input", 0) / 1_000_000 * input_price
    output_cost = usage.get("output", 0) / 1_000_000 * output_price

    return {
    "input": input_cost,
    "output": output_cost,
    "cacheRead": 0.0,
    "cacheWrite": 0.0,
    "total": input_cost + output_cost,
    }

### 重写非 streaming call_model

In [662]:
def call_model_core_actual_usage():
    global LAST_PROVIDER_USAGE
    LAST_PROVIDER_USAGE = None
    
    llm = get_llm_client()
    kwargs = build_model_kwargs()

    try:
        response = llm.chat.completions.create(**kwargs)

    except Exception as e:
        msg = str(e)
        if "reasoning_effort" in msg or "thinking" in msg:
            print("[warning] provider rejected thinking params; retrying without thinking params")
            kwargs.pop("reasoning_effort", None)
            kwargs.pop("extra_body", None)
            response = llm.chat.completions.create(**kwargs)
        else:
            raise

    usage = set_last_provider_usage(getattr(response, "usage", None))

    choice = response.choices[0]
    assistant_msg = choice.message.model_dump(exclude_none=True)

    return annotate_assistant_message(
    assistant_msg,
    usage=usage,
    stop_reason=getattr(choice, "finish_reason", None),
    )


def call_model():
    provider = globals().get("PROVIDER", "unknown")
    model = globals().get("MODEL", "unknown")

    if "call_with_retry" in globals():
        return call_with_retry(
        call_model_core_actual_usage,
        label=f"{provider}/{model}:non_streaming",
        )

    return call_model_core_actual_usage()

### 重写 streaming call_model_streaming

In [663]:
def call_model_streaming_core_actual_usage():
    global LAST_PROVIDER_USAGE
    LAST_PROVIDER_USAGE = None
    
    llm = get_llm_client()
    kwargs = build_model_kwargs()
    kwargs["stream"] = True
    kwargs["stream_options"] = {"include_usage": True}

    try:
        stream = llm.chat.completions.create(**kwargs)

    except Exception as e:
        msg = str(e)

    if "stream_options" in msg or "include_usage" in msg:
        print("[warning] provider rejected stream_options; retrying without usage streaming")
        kwargs.pop("stream_options", None)
        stream = llm.chat.completions.create(**kwargs)

    elif "reasoning_effort" in msg or "thinking" in msg:
        print("[warning] provider rejected thinking params; retrying without thinking params")
        kwargs.pop("reasoning_effort", None)
        kwargs.pop("extra_body", None)
        stream = llm.chat.completions.create(**kwargs)

    else:
        raise

    role = "assistant"
    content_parts = []
    tool_calls_by_index = {}
    finish_reason = None
    
    for chunk in stream:
        usage = getattr(chunk, "usage", None)
        if usage is not None:
            set_last_provider_usage(usage)
        
        if not chunk.choices:
            continue

        choice = chunk.choices[0]
        finish_reason = getattr(choice, "finish_reason", None) or finish_reason
        delta = choice.delta

        if delta is None:
            continue

        d = delta.model_dump(exclude_none=True)

        if d.get("role"):
            role = d["role"]

        if d.get("content"):
            text = d["content"]
            content_parts.append(text)

            if "emit_event" in globals():
                emit_event({
                   "type": "message_update",
                   "assistantMessageEvent": {
                       "type": "text_delta",
                       "delta": text,
                   },
                   "delta": text,
                })
                
                print(text, end="", flush=True)

        for tc in d.get("tool_calls") or []:
            merge_tool_call_delta(tool_calls_by_index, tc)

    content = "".join(content_parts)
    if content:
        print()
    
    assistant_msg = {
    "role": role,
    "content": content if content else None,
    }

    tool_calls = finalize_streamed_tool_calls(tool_calls_by_index)
    if tool_calls:
        assistant_msg["tool_calls"] = tool_calls

    return annotate_assistant_message(
    assistant_msg,
    usage=LAST_PROVIDER_USAGE,
    stop_reason=finish_reason,
    )


def call_model_streaming():
    provider = globals().get("PROVIDER", "unknown")
    model = globals().get("MODEL", "unknown")
    
    if "call_with_retry" in globals():
        return call_with_retry(
        call_model_streaming_core_actual_usage,
        label=f"{provider}/{model}:streaming",
        )
    
    return call_model_streaming_core_actual_usage()

### 重写 call_model_agent

In [664]:
def call_model_agent():
    estimated_input = estimate_messages_tokens(messages)
    
    started = time.time()
    
    if globals().get("STREAMING_ENABLED", True):
        try:
            assistant_msg = call_model_streaming()
            already_rendered = True
        except Exception as e:
            print(f"\n[streaming warning] {type(e).__name__}: {e}")
            print("[streaming warning] fallback to non-streaming call")
            assistant_msg = call_model()
            already_rendered = False
    else:
        assistant_msg = call_model()
        already_rendered = False

    elapsed = time.time() - started

    if not assistant_msg.get("usage"):
        estimated_output = estimate_message_tokens(assistant_msg)
        assistant_msg["usage"] = make_estimated_pi_usage(
        estimated_input,
        estimated_output,
        )

    record_assistant_usage(assistant_msg, elapsed)

    return assistant_msg, already_rendered

### 从 assistant message 记录统计

In [665]:
def record_assistant_usage(assistant_msg, elapsed_seconds=None):
    usage = assistant_msg.get("usage")
    if not usage:
        return None

    provider = assistant_msg.get("provider", globals().get("PROVIDER","unknown"))
    model = assistant_msg.get("model", globals().get("MODEL", "unknown"))
    context_window = get_context_window(provider, model)
    
    rec = {
    "timestamp": assistant_msg.get("timestamp"),
    "provider": provider,
    "model": model,
    "input_tokens": usage.get("input", 0),
    "output_tokens": usage.get("output", 0),
    "cache_read": usage.get("cacheRead", 0),
    "cache_write": usage.get("cacheWrite", 0),
    "reasoning_tokens": usage.get("reasoning", 0),
    "total_tokens": usage.get("totalTokens", 0),
    "context_window": context_window,
    "context_usage_ratio": usage.get("input", 0) / context_window if
    context_window else None,
    "cost_usd": (usage.get("cost") or {}).get("total"),
    "estimated": usage.get("estimated", False),
    "elapsed_seconds": elapsed_seconds,
    }
    
    USAGE_STATS["model_calls"].append(rec)
    USAGE_STATS["total_input_tokens"] += rec["input_tokens"]
    USAGE_STATS["total_output_tokens"] += rec["output_tokens"]

    if rec["cost_usd"] is None:
        USAGE_STATS["has_unknown_cost"] = True
    else:
        USAGE_STATS["total_cost_usd"] += rec["cost_usd"]

    if "emit_event" in globals():
        emit_event({
        "type": "usage_update",
        **rec,
        })

    return rec

### 验证是否拿到真实 usage

In [666]:
new_session()
run("用一句话回答：你是谁？")

New session: D:\workflow\agent_harness\.mini-pi\sessions\20260731-012621-d0f6c989.jsonl
[event] agent_start
[event] turn_start
[event] message_start
[event] agent_end


NameError: name 'USAGE_STATS' is not defined